# Experimento -- Previsao Diaria Hierarquica (`cia_unidade`/`produto` x `canal`)

Tenta bater as metricas dos modelos diarios hoje em producao para o TOTAL
(**ABR-2** para quantidade, **ExtraTrees(BASE,j90)** para valor -- ver `modelo_qtd.py`/
`modelo_valor.py` e a secao "Nowcast recursivo" de `experimento_mensal.ipynb`) fazendo a
previsao de forma **hierarquica**: cada estrato leaf e o cruzamento de
`cia_unidade` (quando o produto e Energia/Luz em Dia) OU `produto` (demais produtos) x
`canal`, somado bottom-up.

Duas estrategias sao comparadas em cada horizonte/alvo:
- **Estrategia A -- melhor modelo por estrato**: cada estrato usa seu proprio vencedor
  (menor RMSPE) entre baselines/regressao/estatisticos/RIPR/Kalman/Blend.
- **Estrategia B -- melhor modelo unico para todas as series**: 1 UNICO modelo global
  (pooled/cross-learning via `mlforecast`, Nixtla), testado com 6/12/24 meses de
  historico de treino.

Uma camada extra de **reconciliacao hierarquica** (`hierarchicalforecast`: BottomUp/
MinTrace/ERM/MiddleOut) entra como comparacao adicional.

> **Nao executado neste ambiente** (sem VPN/kernel) -- as celulas foram escritas seguindo
> os padroes de `experimento_mensal.ipynb`/`experimento_qtd.ipynb`/`experimento_valor.ipynb`,
> mas precisam ser rodadas (com VPN) e iteradas.

# ETL

In [ ]:
# bootstrap: raiz do projeto
import os, pathlib
_r = pathlib.Path.cwd()
while not (_r / 'CLAUDE.md').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r)

## Imports e configuracao

In [ ]:
import copy
import glob
import warnings
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import clickhouse_connect
from workadays import workdays as wd

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor,
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
try:
    from xgboost import XGBRegressor
    _HAS_XGB = True
except Exception:
    _HAS_XGB = False

from statsmodels.tsa.statespace.structural import UnobservedComponents

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoCES, Theta, CrostonSBA, TSB

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean

from hierarchicalforecast.utils import aggregate
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace, ERM, MiddleOut

import sys
sys.path.insert(0, 'Scripts')
from cache_utils import carregar_com_cache

warnings.filterwarnings('ignore')

In [ ]:
# ==================== Configuracao ====================
HOST = '10.101.150.150'
PORT = 8123
USER = 'daniel_ramazzotte'
PASS = 'O9pLTAv*yVz0lGP^#M'

# Historico bruto puxado do ClickHouse -- cobre a maior janela de teste do mlforecast (24
# meses) + aquecimento; ajuste p/ mais se quiser testar janelas maiores.
DIAS_QUERY_HIST = 24 * 31 + 120

JANELAS_HIST_MLFORECAST_MESES = [6, 12, 24]   # teste de janela do modelo pooled (Estrategia B)

MIN_SHARE_CANAL_D   = 0.005   # mesmo limiar (0.5%) do experimento_mensal p/ decidir quem vira
MIN_SHARE_PRODUTO_D = 0.005   # estrato proprio vs cair em 'Outros'
MIN_SHARE_CIA_D     = 0.005

MIN_TREINO_D             = 30    # dias uteis de aquecimento antes do 1o fold do walk-forward por estrato
COBERTURA_MIN_ESTRATO_D  = 0.40  # fracao minima de dias ativos (qntd>0) no periodo p/ o estrato entrar
CRITERIO_ESCOLHA_ESTRATO_D = 'Score'   # 'Score' (composto, ver calcular_score_d) -- alternativas: 'RMSPE (%)' ou 'Vies (real-prev)'

print(f'DIAS_QUERY_HIST={DIAS_QUERY_HIST} | janelas mlforecast={JANELAS_HIST_MLFORECAST_MESES} meses')

## Query hierarquica diaria (`cia_unidade`/`produto` x `canal`)

Mesma composicao de colunas da query `query_painel` de `experimento_mensal.ipynb`
(`ft_proposta` + `ft_contrato` -> `mis_s3.cadastro` p/ canal + `dim_produto` p/
produto/tipoproduto + `dim_convenio` p/ cia/cia_grupo), trocando o agrupamento mensal por
`toDate(ultimaalteracao)` (grao diario) e limitando a janela a `DIAS_QUERY_HIST` (em vez de
"desde 2018") p/ manter a query rapida no grao diario.

In [ ]:
query_painel_diario = f"""
SELECT
    toDate(fp.ultimaalteracao) AS data,
    cad.canal                   AS canal,
    dp.produto                  AS produto,
    dp.tipoproduto                AS tipoproduto,
    dc.convenio                   AS cia,
    dc.conveniogrupo               AS cia_grupo,
    SUM(fp.valor)                   AS valor,
    COUNT(fp.propostaid)            AS qntd
FROM crefaz.ft_proposta fp
LEFT JOIN crefaz.ft_contrato fc  ON fc.propostaid = fp.propostaid
LEFT JOIN mis_s3.cadastro cad     ON cad.loginVendedor = fc.loginvendedorrbm
LEFT JOIN crefaz.dim_produto dp   ON dp.id = fp.produtoid
LEFT JOIN crefaz.dim_convenio dc  ON dc.id = fp.convenioid
WHERE fp.propostaetapaid = 16 AND fp.propostadecisaoid IS NULL
  AND toDate(fp.ultimaalteracao) BETWEEN toDate(today() - INTERVAL {DIAS_QUERY_HIST} DAY) AND toDate(today() - INTERVAL 1 DAY)
GROUP BY data, canal, produto, tipoproduto, cia, cia_grupo
"""
# Canal NAO vem de fp.canalid/dim_canal -- vem do vendedor (mesma ressalva de
# experimento_mensal.ipynb/experimento_qtd.ipynb Secao 6). Cia = dim_convenio (concessionaria
# de energia), joined direto por fp.convenioid.

def _consultar_painel_diario():
    _cli = clickhouse_connect.get_client(host=HOST, port=PORT, username=USER, password=PASS)
    return _cli.query_df(query_painel_diario)

_PATH_SNAPSHOT_PAINEL_D = 'Tabelas/resultados/snapshot_query_diario_hierarquico.xlsx'
try:
    df_painel_d = carregar_com_cache(
        'query_diario_hierarquico',
        [pd.Timestamp.today().date(), DIAS_QUERY_HIST, query_painel_diario],
        _consultar_painel_diario,
    )
except RuntimeError as _e:
    if os.path.exists(_PATH_SNAPSHOT_PAINEL_D):
        print(f'[fallback xlsx] {_e}')
        print(f'-> usando snapshot commitado: {_PATH_SNAPSHOT_PAINEL_D} (pode estar desatualizado)')
        df_painel_d = pd.read_excel(_PATH_SNAPSHOT_PAINEL_D)
    else:
        raise

df_painel_d['data']  = pd.to_datetime(df_painel_d['data'])
df_painel_d['valor'] = df_painel_d['valor'].astype(float)
df_painel_d['qntd']  = df_painel_d['qntd'].astype(float)

print(f'df_painel_d (bruto): {len(df_painel_d):,} linhas | '
      f'{df_painel_d["data"].min().date()} -> {df_painel_d["data"].max().date()}')

In [ ]:
# Fillna de negocio (mesma convencao de experimento_mensal.ipynb):
df_painel_d['canal']       = df_painel_d['canal'].fillna('CORBAN').replace({'LOJAS CREFAZ': 'LOJAS'})
df_painel_d['produto']     = df_painel_d['produto'].fillna('Sem Produto')
df_painel_d['tipoproduto'] = df_painel_d['tipoproduto'].fillna('Nao eletrico')
df_painel_d['cia']         = df_painel_d['cia'].fillna('Sem Cia')
df_painel_d['cia_grupo']   = df_painel_d['cia_grupo'].fillna('Sem Cia')

# Regra de negocio (identica ao mensal, pos-rename cia_unidade): CDC Energia e
# Financiamento Crefaz NAO contam como Eletrico -- competem por estrato de PRODUTO.
df_painel_d['tipoproduto'] = df_painel_d['tipoproduto'].mask(
    df_painel_d['produto'].isin(['CDC Energia', 'Financiamento Crefaz']), 'Nao eletrico')

print('df_painel_d tratado OK')
df_painel_d[['canal', 'produto', 'tipoproduto', 'cia', 'cia_grupo']].nunique()

## Cache de resultados de modelo (walk-forward, trajetoria)

Mesmo espirito do `cache_modelo` de `experimento_mensal.ipynb` (Secoes 6/7): cache em
disco (`Scripts/cache`) dos resultados de walk-forward/trajetoria POR (estrato, modelo,
alvo, horizonte) -- sem isso, cada re-execucao do notebook refaria do zero ~140 estratos x
7 familias de modelo x 2 alvos x 2 horizontes (walk-forward diario refit a cada fold +
nowcast recursivo refit por dia-origem), inviavel na pratica. `chave_partes` sempre inclui
`_versao_dados_d()` (muda quando chega um dia novo de dado), entao o cache se renova
sozinho -- nao precisa limpar manualmente.

In [ ]:
import hashlib as _hashlib_cache
import pickle as _pickle_cache

def cache_modelo_d(prefix, chave_partes, montar, cache_dir='Scripts/cache', forcar=False):
    """Cache em disco p/ resultado de walk-forward/trajetoria (DataFrame, dict, tupla,
    etc.) de UM (estrato, modelo, alvo, horizonte) -- mesma logica de cache_modelo
    (experimento_mensal.ipynb): `chave_partes` DEVE incluir `_versao_dados_d()` p/ nao
    servir resultado desatualizado quando chega dado novo. `montar`: callable sem
    argumentos que roda o walk-forward/trajetoria e retorna o resultado a cachear.
    `forcar=True` (ou RODAR_MODELOS_D=True) ignora o cache e recalcula."""
    os.makedirs(cache_dir, exist_ok=True)
    _chave = _hashlib_cache.md5('|'.join(str(p) for p in chave_partes).encode('utf-8')).hexdigest()[:12]
    _caminho = os.path.join(cache_dir, f'{prefix}_{_chave}.pkl')
    if not forcar and not RODAR_MODELOS_D and os.path.exists(_caminho):
        with open(_caminho, 'rb') as f:
            _resultado = _pickle_cache.load(f)
        return _resultado
    _resultado = montar()
    with open(_caminho, 'wb') as f:
        _pickle_cache.dump(_resultado, f)
    return _resultado


def _versao_dados_d():
    """Componente de chave que muda quando chega um dia novo de dado (df_painel_d ganha
    uma linha) -- mesma logica de _versao_dados (experimento_mensal.ipynb), granularidade
    diaria."""
    return str(df_painel_d['data'].max().date())


RODAR_MODELOS_D = False   # False = usa cache p/ TODOS os candidatos (walk-forward + trajetoria),
                          # recalculando so o que ainda nao tem cache p/ esta _versao_dados_d();
                          # True = ignora cache e recalcula tudo do zero

print(f'cache_modelo_d OK | RODAR_MODELOS_D={RODAR_MODELOS_D} | versao dados: {_versao_dados_d()}')

## Definicao dos estratos (`cia_unidade`/`produto` x `canal`)

In [ ]:
def _top_por_share(df, col, min_share):
    """Nomes de `col` cujo share de qntd >= min_share -- mesmo criterio de
    MIN_SHARE_* em experimento_mensal.ipynb (quem nao entra vira 'Outros')."""
    _tot = df['qntd'].sum()
    if not _tot:
        return []
    _sh = df.groupby(col)['qntd'].sum().sort_values(ascending=False) / _tot
    return sorted(_sh[_sh >= min_share].index.tolist())

_df_eletrico_d    = df_painel_d[df_painel_d['tipoproduto'] == 'Eletrico'].copy()
_df_naoeletrico_d = df_painel_d[df_painel_d['tipoproduto'] != 'Eletrico'].copy()

_cias_unidade_proprias        = _top_por_share(_df_eletrico_d, 'cia', MIN_SHARE_CIA_D)
_produtos_naoenergia_proprios = _top_por_share(_df_naoeletrico_d, 'produto', MIN_SHARE_PRODUTO_D)
_canais_proprios              = _top_por_share(df_painel_d, 'canal', MIN_SHARE_CANAL_D)

print(f'CIA unidade proprias ({len(_cias_unidade_proprias)}):', _cias_unidade_proprias)
print(f'Produtos nao-eletrico proprios ({len(_produtos_naoenergia_proprios)}):', _produtos_naoenergia_proprios)
print(f'Canais proprios ({len(_canais_proprios)}):', _canais_proprios)

In [ ]:
# _MAPA_GRUPO_CIA_UNIDADE: cia_unidade (individual, ex. 'ENEL SP') -> grupo (ex. 'ENEL') --
# usado so pela camada de reconciliacao hierarquica (nivel intermediario 'cia_produto' do
# MiddleOut), MESMA logica de experimento_mensal.ipynb (pos-rename cia_fina -> cia_unidade).
_MAPA_GRUPO_CIA_UNIDADE = (
    _df_eletrico_d[_df_eletrico_d['cia'].isin(_cias_unidade_proprias)]
    .groupby('cia')['cia_grupo'].agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)
print('_MAPA_GRUPO_CIA_UNIDADE:', _MAPA_GRUPO_CIA_UNIDADE)

def _cia_produto_de(cp):
    """Grupo (Eletrico) ou o proprio produto (nao-Eletrico) -- nivel intermediario
    usado so pelo MiddleOut da reconciliacao hierarquica."""
    return _MAPA_GRUPO_CIA_UNIDADE.get(cp, cp) if cp in _cias_unidade_proprias else cp

In [ ]:
# ESTRATOS_CRUZADO_D: leaf = cruzamento (cia_unidade OU produto) x canal -- exatamente a
# hierarquia pedida ("cia_unidade/produto X canal"), mesma composicao de
# ESTRATOS_CRUZADO_UNIDADE (pos-rename) em experimento_mensal.ipynb.
ESTRATOS_CRUZADO_D = [
    ('cruzado', (_cp, _canal))
    for _cp in (_cias_unidade_proprias + _produtos_naoenergia_proprios)
    for _canal in _canais_proprios
]
print(f'ESTRATOS_CRUZADO_D: {len(ESTRATOS_CRUZADO_D)} estratos candidatos (antes do gate de cobertura)')

def _nome_estrato_d(chave):
    _tipo, _nome = chave
    if _tipo == 'cruzado':
        _cp, _canal = _nome
        return f'{_cp} x {_canal}'
    return f'{_tipo}:{_nome}'


def _partes_estrato_d(chave):
    """Quebra a chave em (cia_produto, canal, cia_unidade) p/ o export -- mesma convencao
    de _partes_estrato (experimento_mensal.ipynb, pos-rename): quando `cp` e uma CIA
    INDIVIDUAL, `cia_unidade`=cp e `cia_produto`=grupo dela (via _cia_produto_de); quando
    `cp` e um produto nao-Eletrico, `cia_produto`=cp e `cia_unidade` fica None."""
    _tipo, _nome = chave
    if _tipo != 'cruzado':
        return None, None, None
    _cp, _canal = _nome
    if _cp in _cias_unidade_proprias:
        return _cia_produto_de(_cp), _canal, _cp
    return _cp, _canal, None

## Serie diaria por estrato e engenharia de features

In [ ]:
CALENDARIO_ATIVO_D = pd.date_range(df_painel_d['data'].min(), df_painel_d['data'].max(), freq='D')

def serie_diaria_estrato_d(chave, target):
    """Serie diaria de 1 estrato (cruzado (cia_unidade|produto, canal)) p/ `target`
    ('valor'/'qntd'), reindexada no calendario ATIVO (zero-fill so quando o estrato nao
    teve proposta naquele dia -- nunca dia morto, mesma convencao de montar_ag_canal6 em
    experimento_qtd.ipynb), excluindo domingo e feriado BR."""
    _tipo, _nome = chave
    _cp, _canal = _nome
    if _cp in _cias_unidade_proprias:
        _sub = df_painel_d[(df_painel_d['tipoproduto'] == 'Eletrico') &
                           (df_painel_d['cia'] == _cp) & (df_painel_d['canal'] == _canal)]
    else:
        _sub = df_painel_d[(df_painel_d['tipoproduto'] != 'Eletrico') &
                           (df_painel_d['produto'] == _cp) & (df_painel_d['canal'] == _canal)]
    _ag = _sub.groupby('data')[[target]].sum().reindex(CALENDARIO_ATIVO_D).fillna(0.0)
    _ag.index.name = 'data'
    _hol = pd.Series(_ag.index).apply(lambda d: bool(wd.is_holiday(d.date(), country='BR'))).values
    _valid = (_ag.index.dayofweek != 6) & (~_hol)
    return _ag.loc[_valid, target].astype(float).sort_index()


FEATS_D = ['dia_semana', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6']

def tabela_supervisionada_estrato_d(serie):
    """dia_semana + lag_1..lag_6 do proprio target -- mesma convencao de
    montar_ag_canal6 (experimento_qtd.ipynb), sem lag_7/fila (decisao: fila so existe
    consolidada no Total, omitida no nivel de estrato)."""
    _df = pd.DataFrame({'y': serie})
    _df['dia_semana'] = _df.index.dayofweek
    for _k in range(1, 7):
        _df[f'lag_{_k}'] = _df['y'].shift(_k)
    return _df.dropna()

In [ ]:
# ==================== Gate de cobertura: filtra ESTRATOS_CRUZADO_D pros que tem dado
# suficiente p/ walk-forward proprio (mesmo espirito de COBERTURA_MIN_ESTRATO no mensal,
# adaptado a fracao de dias ATIVOS no periodo, nao mais fracao de dias uteis do mes) ====
def _cobertura_estrato_d(chave, target='qntd'):
    _s = serie_diaria_estrato_d(chave, target)
    if _s.empty:
        return 0.0, 0
    return float((_s > 0).mean()), int(len(_s))

ESTRATOS_VALIDOS_D = []
for _chave in ESTRATOS_CRUZADO_D:
    _cov, _n = _cobertura_estrato_d(_chave)
    if _cov >= COBERTURA_MIN_ESTRATO_D and _n >= MIN_TREINO_D + 10:
        ESTRATOS_VALIDOS_D.append(_chave)

print(f'ESTRATOS_VALIDOS_D: {len(ESTRATOS_VALIDOS_D)} / {len(ESTRATOS_CRUZADO_D)} '
      f'passaram no gate de cobertura (>= {COBERTURA_MIN_ESTRATO_D:.0%} dias ativos, >= {MIN_TREINO_D + 10} dias)')

## Metrica, score composto e agregacao bottom-up (Estrategia A)

Alem das metricas classicas (MAPE/RMSPE/Mediana/Max/R2/Vies), cada candidato recebe um
**score composto** (`calcular_score_d`) que combina 4 leituras complementares de erro --
**RMSPE** (penaliza erros grandes ao quadrado), **% Outliers** (fracao de dias com
MAPE>30%, captura falhas pontuais que a media/RMSPE podem diluir), **Vies %** (SEM modulo
-- tendencia sistematica de sub/superestimar, favorece o extremo mais NEGATIVO/subestimado
do grupo comparado, por decisao explicita) e **Desvio Padrao do Erro %**
(estabilidade/dispersao, independente do vies) -- cada metrica normalizada (min-max)
DENTRO do conjunto de candidatos comparados antes de ponderar, pra nenhuma dominar so pela
escala. `Score` (menor = melhor) e o criterio padrao de escolha automatica
(`CRITERIO_ESCOLHA_ESTRATO_D`).

In [ ]:
def _rank_linha_d(det):
    """Mesmas metricas de _rank_linha (experimento_mensal.ipynb) + as 3 usadas no score
    composto (% Outliers, Vies %, Desvio Padrao Erro %), sobre um DataFrame diario com
    colunas data/Previsto/Realizado."""
    _cols_vazias = {'MAPE Medio (%)': np.nan, 'RMSPE (%)': np.nan, 'Mediana (%)': np.nan,
                    'Max (%)': np.nan, 'R2': np.nan, 'Vies (real-prev)': np.nan,
                    '% Outliers (MAPE>30%)': np.nan, 'Vies (%)': np.nan,
                    'Desvio Padrao Erro (%)': np.nan, 'N': 0}
    _det = det.dropna(subset=['Previsto', 'Realizado'])
    if _det.empty:
        return _cols_vazias
    _r = _det['Realizado'].astype(float)
    _p = _det['Previsto'].astype(float)
    _erro_pct = (_p - _r) / _r.replace(0, np.nan) * 100   # erro % com sinal (+ superestima)
    _ape = _erro_pct.abs()                                  # MAPE por linha
    _dif = _r - _p
    _ssr = float(((_r - _p) ** 2).sum())
    _sst = float(((_r - _r.mean()) ** 2).sum())
    return {
        'MAPE Medio (%)':          round(float(_ape.mean()), 2),
        'RMSPE (%)':                round(float(np.sqrt(np.mean(np.square(_ape.dropna())))), 2),
        'Mediana (%)':              round(float(_ape.median()), 2),
        'Max (%)':                  round(float(_ape.max()), 2),
        'R2':                       round(1 - _ssr / _sst, 3) if _sst else np.nan,
        'Vies (real-prev)':         round(float(_dif.mean()), 1),
        '% Outliers (MAPE>30%)':    round(float((_ape > 30).mean() * 100), 2),
        'Vies (%)':                 round(float(_erro_pct.mean()), 2),
        'Desvio Padrao Erro (%)':   round(float(_erro_pct.std()), 2),
        'N':                        int(len(_det)),
    }


PESOS_SCORE_D = {'RMSPE (%)': 0.35, '% Outliers (MAPE>30%)': 0.25,
                 'Vies (%)': 0.20, 'Desvio Padrao Erro (%)': 0.20}


def calcular_score_d(rank, pesos=None):
    """Score composto (menor = melhor): RMSPE + % Outliers (MAPE>30%) + Vies % (SEM
    modulo -- por decisao explicita, favorece o extremo NEGATIVO/subestimado do grupo
    comparado, nao |vies| perto de zero) + Desvio Padrao do Erro %, cada um normalizado
    (min-max) DENTRO do conjunto de candidatos em `rank` antes de ponderar
    (`PESOS_SCORE_D`) -- senao RMSPE (dezenas) dominaria Vies% (as vezes perto de 0) so
    pela escala. Adiciona a coluna 'Score' e retorna ORDENADO por ela (melhor primeiro)."""
    _pesos = pesos or PESOS_SCORE_D
    _r = rank.copy()
    _norm = pd.DataFrame(index=_r.index)
    for _col in _pesos:
        _serie = pd.to_numeric(_r[_col], errors='coerce')
        _amp = _serie.max() - _serie.min()
        _norm[_col] = (_serie - _serie.min()) / _amp if _amp else 0.0
    _norm = _norm.fillna(1.0)   # candidato sem essa metrica (NaN) penalizado como o PIOR do grupo
    _r['Score'] = sum(_norm[_c] * _p for _c, _p in _pesos.items())
    return _r.sort_values('Score')


def montar_ranking_estrato_d(dets):
    """dets: dict nome_modelo -> DataFrame(data,Previsto,Realizado) de UM estrato.
    Retorna DataFrame indexado por nome_modelo, com a coluna 'Score' e ordenado por
    CRITERIO_ESCOLHA_ESTRATO_D."""
    _linhas = {}
    for _nome, _det in dets.items():
        if _det is None or _det.empty:
            continue
        _linhas[_nome] = _rank_linha_d(_det)
    if not _linhas:
        return pd.DataFrame()
    _rank = calcular_score_d(pd.DataFrame(_linhas).T)
    return _rank.sort_values(CRITERIO_ESCOLHA_ESTRATO_D)


def agregar_estratos_diario_d(dets_por_estrato, rankings_por_estrato, criterio=None):
    """Estrategia A -- 'melhor modelo por estrato': p/ cada estrato usa o proprio
    melhor modelo (menor Score composto, ou RMSPE/|Vies| conforme
    CRITERIO_ESCOLHA_ESTRATO_D) e soma Previsto/Realizado por data -- bottom-up diario,
    mesma logica de agregar_estratos_mensal (experimento_mensal.ipynb), adaptada a grao
    diario."""
    _criterio = criterio or CRITERIO_ESCOLHA_ESTRATO_D
    _linhas, _vencedores = [], {}
    for _chave, _dets in dets_por_estrato.items():
        _rank = rankings_por_estrato.get(_chave)
        if _rank is None or _rank.empty:
            continue
        _melhor = _rank.sort_values(_criterio).index[0]
        if _melhor not in _dets:
            continue
        _vencedores[_nome_estrato_d(_chave)] = _melhor
        _linhas.append(_dets[_melhor][['data', 'Previsto', 'Realizado']])
    if not _linhas:
        return pd.DataFrame(), _vencedores
    _todos = pd.concat(_linhas, ignore_index=True)
    _bu = _todos.groupby('data', as_index=False).agg(Previsto=('Previsto', 'sum'), Realizado=('Realizado', 'sum'))
    return _bu.sort_values('data').reset_index(drop=True), _vencedores

## Export do melhor modelo (funcoes compartilhadas)

Exporta a tabela estratificada (previsto/realizado por estrato + linha Total agregada
bottom-up) do modelo escolhido em CADA horizonte. Por padrao usa o **vencedor automatico**
(`Score`, `calcular_score_d`) de cada estrato -- passando um NOME de modelo do roster
(`modelo_forcado`) forca esse MESMO modelo em todos os estratos que o tiverem (estratos sem
esse modelo ficam de fora, ex.: um estrato esparso que so rodou Croston/TSB nao tem
`'AutoARIMA (statsforecast)'`).

In [ ]:
PASTA_EXPORT_BI_D = os.path.join('Tabelas', 'saidas', 'bi')


def exportar_diario_hierarquico_d(df, nome_arquivo):
    """Grava `df` em Tabelas/saidas/bi/{nome_arquivo} (sheet 'dados'), mesma convencao de
    exportar_bi (experimento_mensal.ipynb) -- caminho fixo, sobrescrito a cada execucao."""
    if df is None or df.empty:
        print(f'[export] {nome_arquivo}: nada a exportar (tabela vazia)')
        return None
    os.makedirs(PASTA_EXPORT_BI_D, exist_ok=True)
    _caminho = os.path.join(PASTA_EXPORT_BI_D, nome_arquivo)
    df.to_excel(_caminho, sheet_name='dados', index=False)
    return _caminho


def montar_export_1p_d(target, dets_por_estrato, vencedores, modelo_forcado=None):
    """Walk-forward 1 passo: 1 linha por (estrato, data) com o modelo ESCOLHIDO (vencedor
    automatico por Score, ou `modelo_forcado` p/ todos os estratos) + 1 linha Total por
    data (bottom-up dos estratos exportados). Colunas: target/cia_produto/canal/
    cia_unidade/modelo/data/previsto/realizado/erro_pct."""
    _linhas, _pulados = [], []
    for _chave, _dets in dets_por_estrato.items():
        _nome_e = _nome_estrato_d(_chave)
        _nome_modelo = modelo_forcado or vencedores.get(_nome_e)
        if _nome_modelo is None or _nome_modelo not in _dets or _dets[_nome_modelo].empty:
            _pulados.append(_nome_e)
            continue
        _cia_produto, _canal, _cia_unidade = _partes_estrato_d(_chave)
        _d = _dets[_nome_modelo][['data', 'Previsto', 'Realizado']].copy()
        _d.insert(0, 'target', target)
        _d.insert(1, 'cia_produto', _cia_produto)
        _d.insert(2, 'canal', _canal)
        _d.insert(3, 'cia_unidade', _cia_unidade)
        _d.insert(4, 'modelo', _nome_modelo)
        _linhas.append(_d)
    if _pulados:
        print(f'[export 1 passo] {len(_pulados)} estrato(s) sem o modelo pedido, pulados: {_pulados[:10]}'
              f'{"..." if len(_pulados) > 10 else ""}')
    if not _linhas:
        return pd.DataFrame()
    _detalhe = pd.concat(_linhas, ignore_index=True)
    _total = _detalhe.groupby('data', as_index=False).agg(Previsto=('Previsto', 'sum'), Realizado=('Realizado', 'sum'))
    _total.insert(0, 'target', target)
    _total.insert(1, 'cia_produto', None)
    _total.insert(2, 'canal', None)
    _total.insert(3, 'cia_unidade', None)
    _total.insert(4, 'modelo', modelo_forcado or 'Estrategia A (melhor por estrato, Score)')
    _out = pd.concat([_total, _detalhe], ignore_index=True)
    _out['erro_pct'] = (_out['Previsto'] - _out['Realizado']) / _out['Realizado'].replace(0, np.nan) * 100
    return _out.rename(columns={'Previsto': 'previsto', 'Realizado': 'realizado'})


def montar_export_traj_d(target, nowcasts_por_estrato, vencedores, modelo_forcado=None):
    """Trajetoria do mes: 1 linha por (estrato, Mes Previsto, Dia Nowcast) com o modelo
    ESCOLHIDO (vencedor automatico por Score, ou `modelo_forcado`) + 1 linha Total por
    (Mes Previsto, Dia Nowcast) bottom-up. Colunas: target/cia_produto/canal/cia_unidade/
    modelo/Mes Previsto/Dia Nowcast/Dias Corridos/Dias Restantes/nowcast/real/erro_pct."""
    _linhas, _pulados = [], []
    for _chave, _nowcasts in nowcasts_por_estrato.items():
        _nome_e = _nome_estrato_d(_chave)
        _nome_modelo = modelo_forcado or vencedores.get(_nome_e)
        if _nome_modelo is None or _nome_modelo not in _nowcasts or _nowcasts[_nome_modelo].empty:
            _pulados.append(_nome_e)
            continue
        _cia_produto, _canal, _cia_unidade = _partes_estrato_d(_chave)
        _cols = ['Mes Previsto', 'Dia Nowcast', 'Dias Corridos', 'Dias Restantes', 'Nowcast', 'Real']
        _d = _nowcasts[_nome_modelo][_cols].copy()
        _d.insert(0, 'target', target)
        _d.insert(1, 'cia_produto', _cia_produto)
        _d.insert(2, 'canal', _canal)
        _d.insert(3, 'cia_unidade', _cia_unidade)
        _d.insert(4, 'modelo', _nome_modelo)
        _linhas.append(_d)
    if _pulados:
        print(f'[export trajetoria] {len(_pulados)} estrato(s) sem o modelo pedido, pulados: {_pulados[:10]}'
              f'{"..." if len(_pulados) > 10 else ""}')
    if not _linhas:
        return pd.DataFrame()
    _detalhe = pd.concat(_linhas, ignore_index=True)
    _total = _detalhe.groupby(['Mes Previsto', 'Dia Nowcast'], as_index=False).agg(
        **{'Dias Corridos': ('Dias Corridos', 'first'), 'Nowcast': ('Nowcast', 'sum'), 'Real': ('Real', 'sum')})
    _total.insert(0, 'target', target)
    _total.insert(1, 'cia_produto', None)
    _total.insert(2, 'canal', None)
    _total.insert(3, 'cia_unidade', None)
    _total.insert(4, 'modelo', modelo_forcado or 'Estrategia A (melhor por estrato, Score)')
    _total['Dias Restantes'] = np.nan
    _cols_final = ['target', 'cia_produto', 'canal', 'cia_unidade', 'modelo', 'Mes Previsto', 'Dia Nowcast',
                   'Dias Corridos', 'Dias Restantes', 'Nowcast', 'Real']
    _out = pd.concat([_total[_cols_final], _detalhe[_cols_final]], ignore_index=True)
    _out['erro_pct'] = (_out['Nowcast'] - _out['Real']) / _out['Real'].replace(0, np.nan) * 100
    return _out.rename(columns={'Nowcast': 'nowcast', 'Real': 'real'})

## Funcoes compartilhadas de modelagem (usadas por `# Valor` e `# Quantidade`)

Todo o motor abaixo e generico em `target` ('valor'/'qntd') e em `chave` (estrato) --
definido uma unica vez aqui e reaproveitado nas secoes `# Valor`/`# Quantidade`, que so
fazem a orquestracao (loop nos estratos + coleta dos resultados) especifica de cada alvo.

#### Baselines

In [ ]:
def wf_baseline_estrato_d(tab, tipo='naive', min_treino=MIN_TREINO_D):
    """Naive (valor do ultimo dia util) ou media movel(5) -- walk-forward expansivo,
    refit implicito a cada fold (sem parametros pra ajustar)."""
    rows = []
    for i in range(min_treino, len(tab)):
        tr, te = tab.iloc[:i], tab.iloc[i]
        if tipo == 'naive':
            p = float(tr['y'].iloc[-1])
        else:
            p = float(tr['y'].iloc[-5:].mean())
        rows.append({'data': te.name, 'Previsto': p, 'Realizado': float(te['y'])})
    return pd.DataFrame(rows)

#### Modelos de regressao (sklearn)

In [ ]:
def _make_ridge():       return Ridge(alpha=1.0)
def _make_rf():          return RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42)
def _make_gbr():         return GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=42)
def _make_extratrees():  return ExtraTreesRegressor(n_estimators=300, max_depth=3, min_samples_leaf=2, random_state=42)
def _make_adaboost():    return AdaBoostRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
def _make_xgb():         return XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                              subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1)

_MODELOS_ARVORE_D = {'Ridge': _make_ridge, 'RandomForest': _make_rf, 'GradientBoosting': _make_gbr,
                     'ExtraTrees': _make_extratrees, 'AdaBoost': _make_adaboost}
if _HAS_XGB:
    _MODELOS_ARVORE_D['XGBoost'] = _make_xgb


def wf_regressao_estrato_d(tab, make_model, min_treino=MIN_TREINO_D):
    """Walk-forward expansivo 1 passo -- refit em CADA fold (mesmo motor de
    TimeSeriesSplit(test_size=1) usado nos notebooks diarios atuais, so que expresso como
    loop explicito pra reaproveitar a MESMA tabela supervisionada em qualquer familia)."""
    rows = []
    for i in range(min_treino, len(tab)):
        tr, te = tab.iloc[:i], tab.iloc[i]
        m = make_model()
        m.fit(tr[FEATS_D].values, tr['y'].values)
        p = float(m.predict(te[FEATS_D].values.reshape(1, -1))[0])
        rows.append({'data': te.name, 'Previsto': p, 'Realizado': float(te['y'])})
    return pd.DataFrame(rows)

#### Modelos estatisticos (statsforecast)

In [ ]:
def _eh_esparsa_d(serie, limiar_zeros=0.30):
    """Mesmo gate de esparsidade de _modelos_statsforecast_para (experimento_mensal.ipynb)
    -- estratos com muitos dias zerados usam Croston/TSB em vez de ARIMA/ETS/CES/Theta."""
    return bool((serie == 0).mean() >= limiar_zeros)


def _modelos_statsforecast_para_d(serie):
    if _eh_esparsa_d(serie):
        return [CrostonSBA(), TSB(alpha_d=0.2, alpha_p=0.2)]
    return [AutoARIMA(season_length=7), AutoETS(season_length=7), AutoCES(season_length=7), Theta(season_length=7)]


def rodar_statsforecast_estrato_d(serie, min_treino=MIN_TREINO_D):
    """Walk-forward 1 passo via StatsForecast.cross_validation (refit=True a cada fold) --
    troca o roster p/ Croston/TSB automaticamente em series esparsas/zero-inflated."""
    _modelos = _modelos_statsforecast_para_d(serie)
    _df = pd.DataFrame({'unique_id': 'serie', 'ds': serie.index, 'y': serie.values})
    _n_windows = len(_df) - min_treino
    if _n_windows < 2:
        return {}
    try:
        _sf = StatsForecast(models=_modelos, freq='D', n_jobs=1)
        _cv = _sf.cross_validation(df=_df, h=1, n_windows=_n_windows, step_size=1, refit=True)
    except Exception as _e:
        print(f'[statsforecast falhou] {_e}')
        return {}
    _out = {}
    for _m in _modelos:
        _nome = type(_m).__name__
        if _nome in _cv.columns:
            _out[f'{_nome} (statsforecast)'] = (
                _cv[['ds', _nome, 'y']].rename(columns={_nome: 'Previsto', 'y': 'Realizado', 'ds': 'data'}))
    return _out

#### RIPR (Ridge/Lasso/ElasticNet, formas funcionais)

In [ ]:
def _formas_ripr_d(tab):
    """3 formas funcionais simples sobre FEATS_D -- reducao do grid RIPR (8 formas) do
    mensal, mantendo o espirito (formas alternativas dos MESMOS lags/dia_semana) num
    volume de fits viavel por estrato em grao diario."""
    _base = tab[FEATS_D].copy()
    _log1p = _base.copy()
    for _c in [c for c in FEATS_D if c.startswith('lag_')]:
        _log1p[_c] = np.log1p(_log1p[_c].clip(lower=0))
    _inter = _base.copy()
    _inter['lag1_x_dsem'] = _inter['lag_1'] * _inter['dia_semana']
    return {'base': _base, 'log1p': _log1p, 'interacao': _inter}


def wf_ripr_estrato_d(tab, min_treino=MIN_TREINO_D):
    """Grid Ridge/Lasso/ElasticNet x 3 formas funcionais -- cada combinacao roda o
    walk-forward inteiro e entra separadamente no ranking do estrato (RIdge com
    Interacoes/Polinomios/Regressao, mesmo espirito do RIPR do mensal)."""
    _formas = _formas_ripr_d(tab)
    _estimadores = {'Ridge': lambda: Ridge(alpha=1.0),
                    'Lasso': lambda: Lasso(alpha=0.01, max_iter=5000),
                    'ElasticNet': lambda: ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000)}
    _out = {}
    for _nf, _Xf in _formas.items():
        _cols = list(_Xf.columns)
        _tab_f = _Xf.copy()
        _tab_f['y'] = tab['y']
        for _ne, _mk in _estimadores.items():
            rows = []
            for i in range(min_treino, len(_tab_f)):
                tr, te = _tab_f.iloc[:i], _tab_f.iloc[i]
                m = _mk()
                m.fit(tr[_cols].values, tr['y'].values)
                p = float(m.predict(te[_cols].values.reshape(1, -1))[0])
                rows.append({'data': te.name, 'Previsto': p, 'Realizado': float(te['y'])})
            _out[f'RIPR({_ne},{_nf})'] = pd.DataFrame(rows)
    return _out

#### Kalman (local linear trend)

In [ ]:
def wf_kalman_estrato_d(serie, min_treino=MIN_TREINO_D):
    """Filtro de Kalman (structural time series, local linear trend) -- refit a cada
    fold, mesma ideia do diagnostico Kalman em experimento_mensal.ipynb, em grao diario."""
    rows = []
    for i in range(min_treino, len(serie)):
        tr, te_data, te_val = serie.iloc[:i], serie.index[i], serie.iloc[i]
        try:
            _mod = UnobservedComponents(tr.values, level='local linear trend')
            _r = _mod.fit(disp=False)
            p = float(np.asarray(_r.forecast(1))[0])
        except Exception:
            p = float(tr.iloc[-1])
        rows.append({'data': te_data, 'Previsto': p, 'Realizado': float(te_val)})
    return pd.DataFrame(rows)

#### Blend/Stacking (SARIMAX/estatistico + Naive)

In [ ]:
def wf_blend_estrato_d(det_estatistico, det_naive, w=0.5):
    """Blend simples: w*estatistico + (1-w)*Naive, alinhado por data (mesmo espirito do
    Blend SARIMAX+Naive do mensal)."""
    _m = det_estatistico.merge(det_naive, on='data', suffixes=('_a', '_b'))
    _m['Previsto'] = w * _m['Previsto_a'] + (1 - w) * _m['Previsto_b']
    _m['Realizado'] = _m['Realizado_a']
    return _m[['data', 'Previsto', 'Realizado']]


def wf_stacking_estrato_d(det_estatistico, det_naive, min_treino=10):
    """Stacking: Ridge treinado sobre (previsto_estatistico, previsto_naive) -> Realizado,
    walk-forward (refit a cada fold usando so os folds ja vistos)."""
    _m = det_estatistico.merge(det_naive, on='data', suffixes=('_a', '_b')).sort_values('data').reset_index(drop=True)
    rows = []
    for i in range(min_treino, len(_m)):
        tr, te = _m.iloc[:i], _m.iloc[i]
        _mdl = Ridge(alpha=1.0)
        _mdl.fit(tr[['Previsto_a', 'Previsto_b']].values, tr['Realizado_a'].values)
        p = float(_mdl.predict([[te['Previsto_a'], te['Previsto_b']]])[0])
        rows.append({'data': te['data'], 'Previsto': p, 'Realizado': float(te['Realizado_a'])})
    return pd.DataFrame(rows)

#### mlforecast (Nixtla, modelo global pooled) -- Estrategia B

In [ ]:
def rodar_mlforecast_pooled_d(target, estratos=None, janela_meses=12, modelos=None, min_treino=MIN_TREINO_D):
    """Estrategia B -- '1 UNICO modelo para todas as series': mlforecast treina 1 MODELO
    GLOBAL (pooled/cross-learning) compartilhado entre TODAS as folhas cruzadas
    (`ESTRATOS_VALIDOS_D`), diferente do roster acima que treina 1 modelo POR estrato.
    `janela_meses`: quantos meses de historico recente entram no treino (testado com
    6/12/24 -- ver JANELAS_HIST_MLFORECAST_MESES)."""
    modelos = modelos or {'Ridge': Ridge(alpha=1.0),
                          'ExtraTrees': ExtraTreesRegressor(n_estimators=300, max_depth=3, random_state=42)}
    _estratos = estratos if estratos is not None else ESTRATOS_VALIDOS_D
    _corte = CALENDARIO_ATIVO_D.max() - pd.DateOffset(months=janela_meses)
    _paineis = []
    for _chave in _estratos:
        _s = serie_diaria_estrato_d(_chave, target)
        _s = _s[_s.index >= _corte]
        if len(_s) < min_treino + 5:
            continue
        _paineis.append(pd.DataFrame({'unique_id': _nome_estrato_d(_chave), 'ds': _s.index, 'y': _s.values}))
    if not _paineis:
        return {}, pd.DataFrame()
    _painel = pd.concat(_paineis, ignore_index=True)
    _tam_min = int(_painel.groupby('unique_id').size().min())
    _n_windows = _tam_min - min_treino
    if _n_windows < 2:
        return {}, _painel
    _mlf = MLForecast(models=modelos, freq='D', lags=[1, 2, 3, 4, 5, 6],
                      lag_transforms={1: [RollingMean(window_size=3)], 2: [RollingMean(window_size=7)]},
                      date_features=['dayofweek'])
    _cv = _mlf.cross_validation(df=_painel, n_windows=_n_windows, h=1, step_size=1, refit=True)
    _out = {}
    for _nome_m in modelos:
        if _nome_m in _cv.columns:
            _out[_nome_m] = _cv[['unique_id', 'ds', _nome_m, 'y']].rename(
                columns={_nome_m: 'Previsto', 'y': 'Realizado', 'ds': 'data'})
    return _out, _painel


def escolher_janela_mlforecast_d(target, estratos=None, modelos=None):
    """Roda rodar_mlforecast_pooled_d p/ cada janela em JANELAS_HIST_MLFORECAST_MESES,
    soma bottom-up por janela e retorna (melhor_janela, dict janela->(dets, bu, rank))."""
    _resultados = {}
    for _jm in JANELAS_HIST_MLFORECAST_MESES:
        _dets, _painel = rodar_mlforecast_pooled_d(target, estratos=estratos, janela_meses=_jm, modelos=modelos)
        if not _dets:
            continue
        for _nome_m, _det_pooled in _dets.items():
            _bu = _det_pooled.groupby('data', as_index=False).agg(Previsto=('Previsto', 'sum'), Realizado=('Realizado', 'sum'))
            _rank = _rank_linha_d(_bu)
            _resultados[(_jm, _nome_m)] = {'dets': _dets, 'painel': _painel, 'bu': _bu, 'rank': _rank}
    if not _resultados:
        return None, {}
    _melhor_chave = min(_resultados, key=lambda k: _resultados[k]['rank'].get('RMSPE (%)', np.inf))
    return _melhor_chave, _resultados

#### Reconciliacao hierarquica (hierarchicalforecast) -- camada extra de comparacao

In [ ]:
SPEC_HIERARQUICO_D = [['total'], ['total', 'cia_produto'], ['total', 'cia_produto', 'cia_unidade'],
                      ['total', 'canal'], ['total', 'cia_produto', 'cia_unidade', 'canal']]


def _painel_hierarquico_d(target, estratos=None):
    """Monta o painel long (total/cia_produto/cia_unidade/canal/ds/y) no formato exigido
    por hierarchicalforecast.utils.aggregate -- mesma convencao (pass-through cia_unidade
    = cia_produto p/ produto sem CIA de verdade) de experimento_mensal.ipynb."""
    _estratos = estratos if estratos is not None else ESTRATOS_VALIDOS_D
    _linhas = []
    for _chave in _estratos:
        _tipo, (_cp, _canal) = _chave
        _s = serie_diaria_estrato_d(_chave, target)
        if _s.empty:
            continue
        _cia_produto = _cia_produto_de(_cp)
        _cia_unidade_h = _cp if _cp in _cias_unidade_proprias else _cia_produto
        _linhas.append(pd.DataFrame({'total': 'total', 'cia_produto': _cia_produto,
                                     'cia_unidade': _cia_unidade_h, 'canal': _canal,
                                     'ds': _s.index, 'y': _s.values}))
    return pd.concat(_linhas, ignore_index=True) if _linhas else pd.DataFrame()


def rodar_reconciliacao_hierarquica_d(target, estratos=None, min_treino=MIN_TREINO_D):
    """4 reconciliadores (BottomUp/MinTrace/ERM na hierarquia agrupada completa,
    MiddleOut na sub-hierarquia sem Canal) -- base forecast comum: AutoARIMA(season_length=7)
    em TODOS os nos, walk-forward 1 passo (refit por dia-origem)."""
    _painel = _painel_hierarquico_d(target, estratos=estratos)
    if _painel.empty:
        return {}
    _Y_df, _S_df, _tags = aggregate(df=_painel, spec=SPEC_HIERARQUICO_D)
    _datas = sorted(_Y_df['ds'].unique())
    if len(_datas) < min_treino + 5:
        return {}
    _hrec = HierarchicalReconciliation(reconcilers=[BottomUp(), MinTrace(method='mint_shrink'), ERM(method='reg_bu')])
    _linhas = {}
    for _d_alvo in _datas[min_treino:]:
        _treino = _Y_df[_Y_df['ds'] < _d_alvo]
        try:
            _sf = StatsForecast(models=[AutoARIMA(season_length=7)], freq='D', n_jobs=1)
            _fc = _sf.forecast(df=_treino, h=1, fitted=True)
            _fit = _sf.forecast_fitted_values()
            _rec = _hrec.reconcile(Y_hat_df=_fc, Y_df=_fit, S_df=_S_df, tags=_tags)
        except Exception:
            continue
        _real = float(_Y_df[(_Y_df['unique_id'] == 'total') & (_Y_df['ds'] == _d_alvo)]['y'].sum())
        for _col in ['AutoARIMA/BottomUp', 'AutoARIMA/MinTrace_method-mint_shrink', 'AutoARIMA/ERM_method-reg_bu']:
            if _col in _rec.columns:
                _p = float(_rec.loc[_rec['unique_id'] == 'total', _col].iloc[0])
                _linhas.setdefault(_col, []).append({'data': _d_alvo, 'Previsto': _p, 'Realizado': _real})
    return {k: pd.DataFrame(v) for k, v in _linhas.items()}

## Motor generico de trajetoria (nowcast recursivo por estrato)

Generaliza `nowcast_tabela_estrato`/`nowcast_tabela_ml_recursivo_valor`/`_qtd`
(`experimento_mensal.ipynb`): p/ cada dia util D do mes-alvo, soma o REALIZADO ate D com a
previsao dos dias uteis RESTANTES do mes, refeita do zero em CADA dia-origem D (so com
dados <= D). O que muda por familia de modelo e SO a funcao `prever_multi_passo` (direta
via horizonte nativo p/ statsforecast/mlforecast, ou recursiva -- previsao vira lag da
proxima -- p/ sklearn/RIPR/baselines/Kalman).

In [ ]:
def nowcast_generico_estrato_d(serie, prever_multi_passo, ano, mes, min_treino=MIN_TREINO_D):
    """`prever_multi_passo(serie_ate_D, dias_alvo) -> lista de previsoes (mesma ordem de
    dias_alvo)` -- plugavel por familia de modelo (ver builders abaixo)."""
    _idx = pd.DatetimeIndex(serie.index)
    _dias_mes = sorted(_idx[(_idx.year == ano) & (_idx.month == mes)])
    if not _dias_mes:
        return pd.DataFrame()
    _total_real = float(serie.loc[_dias_mes].sum())
    rows = []
    for D in _dias_mes:
        _elapsed = [d for d in _dias_mes if d <= D]
        _remaining = [d for d in _dias_mes if d > D]
        _real_ate_D = float(serie.loc[_elapsed].sum())
        _prev_rest = 0.0
        if _remaining:
            _serie_ate_D = serie.loc[:D]
            if len(_serie_ate_D) < min_treino:
                _prev_rest = np.nan
            else:
                try:
                    _preds = prever_multi_passo(_serie_ate_D, _remaining)
                    _prev_rest = float(np.nansum(np.asarray(_preds, dtype=float)))
                except Exception:
                    _prev_rest = np.nan
        _nowcast_total = _real_ate_D + _prev_rest
        rows.append({'Dia Nowcast': pd.Timestamp(D).normalize(), 'Dias Corridos': len(_elapsed),
                     'Dias Restantes': len(_remaining), 'Nowcast': round(_nowcast_total, 1),
                     'Real': round(_total_real, 1),
                     'Erro %': (round((_nowcast_total - _total_real) / _total_real * 100, 2)
                                if _total_real else np.nan)})
    return pd.DataFrame(rows)


def _meses_completos_serie_d(serie):
    """(ano, mes) dos meses COMPLETOS numa serie diaria -- mesma logica de
    _meses_completos_serie (experimento_mensal.ipynb)."""
    _idx = pd.DatetimeIndex(serie.index)
    _max = serie.index.max()
    out = []
    for (a, m) in sorted({(d.year, d.month) for d in _idx}):
        _pres = sorted(_idx[(_idx.year == a) & (_idx.month == m)])
        _ini = pd.Timestamp(a, m, 1)
        _fim = _ini + pd.offsets.MonthEnd(0)
        _cal = [d for d in pd.bdate_range(_ini, _fim) if not wd.is_holiday(d.date(), country='BR')]
        if _pres and len(_pres) == len(_cal) and pd.Timestamp(_pres[-1]) <= _max:
            out.append((a, m))
    return out


def concatenar_nowcast_multi_mes_d(fn_mes, meses):
    """Concatena o nowcast de VARIOS meses -- mesma funcao de experimento_mensal.ipynb
    (concatenar_nowcast_multi_mes), reaproveitada tal como esta."""
    frames = []
    for (a, m) in meses:
        df = fn_mes(a, m)
        if df is None or df.empty:
            continue
        d = df.copy()
        d['Mes Previsto'] = f'{a:04d}-{m:02d}'
        d['Razao'] = pd.to_numeric(d['Nowcast'], errors='coerce') / pd.to_numeric(d['Real'], errors='coerce')
        frames.append(d)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def plot_nowcast_mediana_erro_d(df_multi, titulo, unidade='', min_meses=2, figsize=(14, 10)):
    """Trajetoria MEDIANA do erro % do nowcast por dia corrido do mes, com banda
    interquartilica -- mesma funcao de experimento_mensal.ipynb (plot_nowcast_mediana_erro)."""
    if df_multi is None or df_multi.empty:
        print(f'[{titulo}] sem dados'); return df_multi
    d = df_multi.dropna(subset=['Razao']).copy()
    d['Erro %'] = (d['Razao'] - 1.0) * 100.0
    d['|Erro %|'] = d['Erro %'].abs()
    _n_mes = d['Mes Previsto'].nunique()

    def _stats(col):
        g = d.groupby('Dias Corridos')[col]
        s = pd.DataFrame({'n': g.size(), 'med': g.median(), 'q1': g.quantile(0.25), 'q3': g.quantile(0.75)}).reset_index()
        return s[s['n'] >= min_meses]

    st_abs, st_sgn = _stats('|Erro %|'), _stats('Erro %')
    if st_abs.empty:
        print(f'[{titulo}] nenhum dia com >= {min_meses} meses'); return d

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)
    cor = '#1f77b4'
    ax1.fill_between(st_abs['Dias Corridos'], st_abs['q1'], st_abs['q3'], color=cor, alpha=0.20, label='Intervalo interquartil (25-75%)')
    ax1.plot(st_abs['Dias Corridos'], st_abs['med'], color=cor, lw=2.2, marker='o', ms=5, label=f'{unidade or "erro"} (mediana)')
    ax1.set_title(f'{titulo} -- |Erro %| (mediana, N={_n_mes} meses)', fontweight='bold')
    ax1.grid(alpha=0.3); ax1.legend(loc='best', fontsize=9)
    ax2.axhline(0.0, color='#555', lw=1.4, ls='--', label='sem vies (erro = 0%)')
    ax2.fill_between(st_sgn['Dias Corridos'], st_sgn['q1'], st_sgn['q3'], color=cor, alpha=0.20)
    ax2.plot(st_sgn['Dias Corridos'], st_sgn['med'], color=cor, lw=2.2, marker='o', ms=5)
    ax2.set_title(f'{titulo} -- Erro % com sinal (+ superestima / - subestima)', fontweight='bold')
    ax2.set_xlabel('Dia util do mes (dias corridos)'); ax2.grid(alpha=0.3); ax2.legend(loc='best', fontsize=9)
    plt.tight_layout(); plt.show()
    return {'abs': st_abs, 'sinal': st_sgn}


def plot_nowcast_trajetoria_estrato_d(df_nc, titulo, unidade='R$', figsize=(12, 5)):
    """Diagnostico de UM mes/estrato: linha CONSTANTE = total real do mes; trajetoria =
    nowcast por dia-origem (deve convergir ao real conforme o mes avanca), com banda +-5%
    -- mesma funcao de experimento_mensal.ipynb (plot_nowcast_trajetoria_estrato)."""
    if df_nc is None or df_nc.empty:
        print(f'[{titulo}] sem dados p/ plotar'); return
    x = pd.to_datetime(df_nc['Dia Nowcast'])
    y = pd.to_numeric(df_nc['Nowcast'], errors='coerce')
    real = float(pd.to_numeric(df_nc['Real'], errors='coerce').iloc[0])
    fig, ax = plt.subplots(figsize=figsize)
    ax.axhspan(real * 0.95, real * 1.05, color='#2ca02c', alpha=0.10, label='+-5% do real')
    ax.axhline(real, color='#2ca02c', lw=2.2, label=f'Real do mes = {real:,.0f}')
    ax.plot(x, y, color='#1f77b4', lw=1.8, marker='o', ms=5, label='Nowcast (total do mes)')
    for _xi, _yi, _e in zip(x, y, pd.to_numeric(df_nc['Erro %'], errors='coerce')):
        if pd.notna(_e):
            ax.annotate(f'{_e:+.0f}%', (_xi, _yi), textcoords='offset points',
                        xytext=(0, 7), ha='center', fontsize=7, color='#555')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Dia em que o nowcast foi feito'); ax.set_ylabel(unidade)
    ax.legend(loc='best', fontsize=9); ax.grid(alpha=0.3)
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()


def plot_nowcast_timeline_d(df_multi, titulo, unidade='', desde=None, prev_d1_por_mes=None, figsize=(16, 7)):
    """Timeline continua no CALENDARIO (varios meses em sequencia): UMA linha do nowcast
    ao longo dos dias-origem + o realizado de cada mes como degrau constante, com
    marcacao do maior erro +/- por mes -- mesma funcao de experimento_mensal.ipynb
    (plot_nowcast_timeline)."""
    if df_multi is None or df_multi.empty:
        print(f'[{titulo}] sem dados'); return df_multi
    d = df_multi.copy()
    d['Dia Nowcast'] = pd.to_datetime(d['Dia Nowcast'])
    if desde is not None:
        d = d[d['Dia Nowcast'] >= pd.Timestamp(desde[0], desde[1], 1)]
    if d.empty:
        print(f'[{titulo}] sem dados a partir de {desde}'); return d
    d = d.sort_values('Dia Nowcast')
    x = d['Dia Nowcast']; y = pd.to_numeric(d['Nowcast'], errors='coerce')
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(x, y, color='#1f77b4', lw=1.8, marker='o', ms=3.5, label='Nowcast (total do mes corrente)')
    _lbl_d1 = False
    for k, mes in enumerate(list(dict.fromkeys(d['Mes Previsto']))):
        g = d[d['Mes Previsto'] == mes]
        real = float(pd.to_numeric(g['Real'], errors='coerce').iloc[0])
        _x0, _x1 = g['Dia Nowcast'].min(), g['Dia Nowcast'].max()
        ax.hlines(real, _x0, _x1, color='#2ca02c', lw=2.6, label='Realizado do mes (degrau)' if k == 0 else None)
        if prev_d1_por_mes and mes in prev_d1_por_mes and np.isfinite(prev_d1_por_mes[mes]):
            ax.hlines(prev_d1_por_mes[mes], _x0, _x1, color='#9467bd', lw=2.0, ls=':',
                      label='Previsao d-1 (fim do mes anterior)' if not _lbl_d1 else None)
            _lbl_d1 = True
        gg = g.assign(_e=pd.to_numeric(g['Erro %'], errors='coerce')).dropna(subset=['_e'])
        _marca = []
        _pos, _neg = gg[gg['_e'] > 0], gg[gg['_e'] < 0]
        if not _pos.empty:
            _marca.append(_pos.loc[_pos['_e'].idxmax()])
        if not _neg.empty:
            _marca.append(_neg.loc[_neg['_e'].idxmin()])
        for r in _marca:
            _xi, _yi, _e = r['Dia Nowcast'], float(pd.to_numeric(r['Nowcast'])), float(r['_e'])
            _c = '#c0392b' if _e > 0 else '#2471a3'
            ax.plot([_xi], [_yi], marker='o', ms=7, color=_c, zorder=5)
            ax.annotate(f'{_e:+.1f}%', (_xi, _yi), textcoords='offset points',
                        xytext=(0, 9 if _e > 0 else -9), ha='center',
                        va='bottom' if _e > 0 else 'top', fontsize=7.5,
                        fontweight='bold', color=_c)
    ax.set_title(f'{titulo}\n(linha unica = nowcast do total do mes corrente | degraus verdes = realizado de cada mes)',
                fontweight='bold')
    ax.set_xlabel('Data (dia em que o nowcast foi feito)'); ax.set_ylabel(unidade or '')
    ax.grid(alpha=0.3); ax.legend(loc='best', fontsize=9)
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()
    return d

### Builders de `prever_multi_passo` por familia de modelo

In [ ]:
def _builder_recursivo_sklearn_d(make_model):
    """sklearn/RIPR: recursao manual -- cada previsao vira o lag_1 do proximo passo
    (empurrando lag_2..lag_6), dia_semana conhecido de antemao (mesmo padrao de
    _recursivo_valor/_recursivo_qtd em experimento_mensal.ipynb)."""
    def _fn(serie_ate_D, dias_alvo):
        _tab = tabela_supervisionada_estrato_d(serie_ate_D)
        _m = make_model()
        _m.fit(_tab[FEATS_D].values, _tab['y'].values)
        _hist = list(serie_ate_D.iloc[-6:].values.astype(float))
        preds = []
        for _d in dias_alvo:
            _row = {'dia_semana': _d.weekday()}
            for _k in range(1, 7):
                _row[f'lag_{_k}'] = _hist[-_k]
            _X = pd.DataFrame([_row])[FEATS_D]
            _p = float(_m.predict(_X.values)[0])
            preds.append(_p)
            _hist.append(_p)
        return preds
    return _fn


def _builder_baseline_d(tipo='naive'):
    def _fn(serie_ate_D, dias_alvo):
        _base = float(serie_ate_D.iloc[-1]) if tipo == 'naive' else float(serie_ate_D.iloc[-5:].mean())
        return [_base] * len(dias_alvo)
    return _fn


def _builder_kalman_d():
    def _fn(serie_ate_D, dias_alvo):
        _mod = UnobservedComponents(serie_ate_D.values, level='local linear trend')
        _r = _mod.fit(disp=False)
        return list(np.asarray(_r.forecast(len(dias_alvo))))
    return _fn


def _builder_statsforecast_d(modelo_cls, **kw):
    """Horizonte NATIVO (h=len(dias_alvo)) -- direto, sem recursao manual."""
    def _fn(serie_ate_D, dias_alvo):
        _df = pd.DataFrame({'unique_id': 's', 'ds': serie_ate_D.index, 'y': serie_ate_D.values})
        _sf = StatsForecast(models=[modelo_cls(**kw)], freq='D', n_jobs=1)
        _fc = _sf.forecast(df=_df, h=len(dias_alvo))
        return list(_fc.iloc[:, -1].values[:len(dias_alvo)])
    return _fn

## Benchmarks de producao (Total) -- ABR-2 (qtd) / ExtraTrees(BASE,j90) (valor)

Reaproveita, quase textualmente, o loader do ABR-2 e a receita do ExtraTrees(BASE,j90) ja
implementados em `experimento_mensal.ipynb` (celulas "ABR-2 (qtd): shims de unpickling
TPOT" e "Motor de previsao RECURSIVA") -- e a reproducao fiel da producao real, entao usa
`lag_7`/fila (diferente dos estratos novos, que omitem `lag_7`).

In [ ]:
# ==================== ABR-2 (qtd): shims de unpickling TPOT + carregamento pelo .pkl real
# (identico a experimento_mensal.ipynb/experimento_qtd.ipynb -- versoes antigas do TPOT
# referenciam classes que nao existem na API atual; sem os shims, joblib.load falha) =====
import sys as _sys, types as _types, joblib as _joblib

try:
    import sklearn._loss._loss as _sll
    def _sll_getattr(name):
        if name.startswith('__pyx_unpickle_'):
            def _unpickle(cls, checksum, state, **kw):
                try:
                    kw2 = {k: v for k, v in state.items()} if isinstance(state, dict) else {}
                    return cls(**kw2)
                except Exception:
                    try:    return cls.__new__(cls)
                    except Exception: return object.__new__(cls)
            return _unpickle
        raise AttributeError(name)
    _sll.__getattr__ = _sll_getattr
except Exception:
    pass

class _ZeroCount:
    def fit(self, X, y=None): return self
    def transform(self, X):
        Xa = X.values if hasattr(X, 'values') else np.asarray(X)
        n0 = (Xa == 0).sum(axis=1, keepdims=True).astype(float)
        n1 = (Xa != 0).sum(axis=1, keepdims=True).astype(float)
        return np.hstack([Xa, n0, n1])
    def fit_transform(self, X, y=None): return self.fit(X, y).transform(X)

class _OHE:
    def __init__(self, minimum_fraction=0.05, sparse=True, threshold=10, **kw):
        self.threshold = threshold; self._cat = []; self._ohe = None
    def fit(self, X, y=None):
        from sklearn.preprocessing import OneHotEncoder as _SOHE
        Xa = X.values if hasattr(X, 'values') else np.asarray(X)
        self._cat = [i for i in range(Xa.shape[1]) if len(np.unique(Xa[:, i])) <= self.threshold]
        self._num = [i for i in range(Xa.shape[1]) if i not in self._cat]
        if self._cat:
            self._ohe = _SOHE(sparse_output=False, handle_unknown='ignore')
            self._ohe.fit(Xa[:, self._cat])
        return self
    def transform(self, X):
        Xa = X.values if hasattr(X, 'values') else np.asarray(X)
        parts = [Xa[:, self._num]] if self._num else []
        if self._cat and self._ohe:
            parts.append(self._ohe.transform(Xa[:, self._cat]))
        return np.hstack(parts) if parts else Xa
    def fit_transform(self, X, y=None): return self.fit(X, y).transform(X)

class _SE:
    def __init__(self, estimator=None, **kw): self.estimator = estimator
    def fit(self, X, y=None):
        if self.estimator: self.estimator.fit(X, y)
        return self
    def transform(self, X):
        Xa = X.values if hasattr(X, 'values') else np.asarray(X, dtype=float)
        if self.estimator:
            p = np.atleast_1d(self.estimator.predict(Xa)).reshape(-1, 1)
            return np.hstack([Xa, p])
        return Xa
    def fit_transform(self, X, y=None): return self.fit(X, y).transform(X)

class _Pass:
    def fit(self, X, y=None): return self
    def transform(self, X): return X
    def fit_transform(self, X, y=None): return X

class _FSS:
    def __init__(self, sel_subset=0, feat_list=None, feat_dict=None, **kw):
        self.sel_subset = sel_subset; self.feat_list = feat_list or []; self.feat_dict = feat_dict or {}
    def fit(self, X, y=None): return self
    def transform(self, X):
        Xa = np.array(X.values if hasattr(X, 'values') else X, dtype=float)
        if not self.feat_dict:
            return Xa
        keys = list(self.feat_dict.keys())
        if isinstance(self.sel_subset, int) and self.sel_subset < len(keys):
            indices = self.feat_dict[keys[self.sel_subset]]
        elif isinstance(self.sel_subset, str) and self.sel_subset in self.feat_dict:
            indices = self.feat_dict[self.sel_subset]
        elif isinstance(self.sel_subset, (list, tuple)):
            indices = list(self.sel_subset)
        else:
            return Xa
        indices = [i for i in indices if i < Xa.shape[1]]
        return Xa[:, indices] if indices else Xa
    def fit_transform(self, X, y=None): return self.fit(X, y).transform(X)

class _PassThru:
    def __init__(self, **kw): pass
    def fit(self, X, y=None): return self
    def transform(self, X): return X.values if hasattr(X, 'values') else np.asarray(X, dtype=float)
    def fit_transform(self, X, y=None): return self.fit(X, y).transform(X)

_shim_defs = {
    'tpot.builtins':                                            {'StackingEstimator': _SE, 'ZeroCount': _ZeroCount, 'OneHotEncoder': _OHE},
    'tpot.builtins.stacking_estimator':                         {'StackingEstimator': _SE},
    'tpot.builtins.zero_count':                                 {'ZeroCount': _ZeroCount},
    'tpot.builtins.one_hot_encoder':                            {'OneHotEncoder': _OHE},
    'tpot.builtin_modules':                                     {'FeatureSetSelector': _FSS},
    'tpot.builtin_modules.passthrough':                         {'Passthrough': _Pass, 'SkipTransformer': _Pass},
    'tpot.builtin_modules.feature_set_selector':                {'FeatureSetSelector': _FSS},
    'tpot.builtin_modules.arithmetic_transformer':              {'ArithmeticTransformer': _PassThru, 'AddTransformer': _PassThru, 'mul_neg_1_Transformer': _PassThru},
    'tpot.builtin_modules.genetic_encoders':                    {'GeneticEncoderClassification': _PassThru, 'GeneticEncoderRegression': _PassThru, 'GeneticEncoder': _PassThru},
    'tpot.builtin_modules.feature_encoding_frequency_selector': {'FeatureEncodingFrequencySelector': _PassThru},
    'tpot.builtin_modules.nn':                                  {'PytorchLRClassifier': _PassThru, 'PytorchMLPClassifier': _PassThru},
    'tpot.builtin_modules.feature_transformers':                {'ContinuousSelector': _PassThru, 'CategoricalSelector': _PassThru},
}
for _mn, _attrs in _shim_defs.items():
    if _mn not in _sys.modules:
        _m = _types.ModuleType(_mn); _m.__path__ = []
        _sys.modules[_mn] = _m
    else:
        _m = _sys.modules[_mn]
    for _k, _v in _attrs.items():
        setattr(_m, _k, _v)
    _m.__getattr__ = lambda name: _PassThru
_bm = _sys.modules['tpot.builtin_modules']
for _sub in ['passthrough', 'feature_set_selector', 'arithmetic_transformer',
             'genetic_encoders', 'feature_encoding_frequency_selector', 'nn', 'feature_transformers']:
    setattr(_bm, _sub, _sys.modules[f'tpot.builtin_modules.{_sub}'])


def _pipeline_sigla_d(pipeline, _S={
    'GradientBoostingRegressor': 'GBR', 'XGBRegressor': 'XGB',
    'RandomForestRegressor': 'RFR',     'ExtraTreesRegressor': 'ETR',
    'Ridge': 'RID', 'Lasso': 'LAS',    'ElasticNet': 'EN',
    'LinearRegression': 'LR',           'HuberRegressor': 'HUB',
    'SVR': 'SVR', 'KNeighborsRegressor': 'KNR',
    'DecisionTreeRegressor': 'DTR',     'AdaBoostRegressor': 'ABR',
    'BaggingRegressor': 'BAG',          'ARDRegression': 'ARD',
    'StandardScaler': 'SS',             'RobustScaler': 'RS',
    'MaxAbsScaler': 'MAS',              'MinMaxScaler': 'MMS',
    'Normalizer': 'NRM', 'PCA': 'PCA', 'SelectKBest': 'SKB',
    'SelectPercentile': 'SPct',         'PolynomialFeatures': 'POLY',
    'FastICA': 'ICA', 'VarianceThreshold': 'VT', 'Binarizer': 'BIN',
    '_SE': 'SE',   'StackingEstimator': 'SE',
    '_ZeroCount': 'ZC', 'ZeroCount': 'ZC',
    '_OHE': 'OHE', 'OneHotEncoder': 'OHE',
    '_FSS': 'FSS', 'FeatureSetSelector': 'FSS',
    '_Pass': 'PASS', 'Passthrough': 'PASS', '_PassThru': 'PASS',
}):
    """Mesma abreviacao de classe->sigla de experimento_qtd.ipynb -- reproduzida aqui p/
    localizar o mesmo .pkl que aquele notebook chama de 'ABR-2'."""
    def _cls(est):
        c = type(est).__name__
        s = _S.get(c, c[:4].upper())
        if c in ('_SE', 'StackingEstimator') and getattr(est, 'estimator', None) is not None:
            ic = type(est.estimator).__name__
            s = f"SE({_S.get(ic, ic[:4].upper())})"
        return s
    if hasattr(pipeline, 'steps'):
        return '+'.join(_cls(e) for _, e in pipeline.steps)
    return _cls(pipeline)


def _carregar_pipeline_abr2_qtd_d():
    """Enumera Tabelas/modelos/quantidade/modelo_treinado-*.pkl em ordem de data, calcula
    a sigla de cada um e retorna o pipeline cuja sigla resolve p/ 'ABR-2' (2a colisao da
    sigla 'ABR') -- MESMA logica de experimento_mensal.ipynb/experimento_qtd.ipynb."""
    _pasta_mod = os.path.join('Tabelas', 'modelos', 'quantidade')
    _pkls_raw = sorted(glob.glob(os.path.join(_pasta_mod, 'modelo_treinado-*.pkl')))
    _entries = []
    for _p in _pkls_raw:
        _stem = os.path.basename(_p).replace('modelo_treinado-', '').replace('.pkl', '').strip()
        if 'opia' in _stem:
            continue
        try:
            _entries.append((pd.Timestamp(_stem), _p))
        except Exception:
            pass
    _entries.sort(key=lambda x: x[0])
    _cnt = 0
    for _dt, _path in _entries:
        try:
            _pip = _joblib.load(_path)
        except Exception:
            continue
        if _pipeline_sigla_d(_pip) == 'ABR':
            _cnt += 1
            if _cnt == 2:
                print(f'[ABR-2] carregado de {_path} ({_dt.date()})')
                return _pip
    raise RuntimeError('ABR-2 nao encontrado entre os .pkl de Tabelas/modelos/quantidade')


try:
    _PIPELINE_ABR2_QTD = _carregar_pipeline_abr2_qtd_d()
except Exception as _e:
    _PIPELINE_ABR2_QTD = None
    print(f'[ABR-2 indisponivel] {_e} -- benchmark de producao de quantidade sera pulado')

In [ ]:
# ==================== Series TOTAIS (dia_semana + lag_0..lag_7) usadas SO pelos
# benchmarks de producao -- vem do proprio df_painel_d somado por data (equivalente ao
# df_diario da Secao 1 dos notebooks diarios atuais) =====================================
_df_total_d = df_painel_d.groupby('data', as_index=False).agg(valor=('valor', 'sum'), qntd=('qntd', 'sum'))
serie_valor_total_d = _df_total_d.set_index('data')['valor'].astype(float).sort_index()
serie_qntd_total_d  = _df_total_d.set_index('data')['qntd'].astype(float).sort_index()

# Fila (etapa 15) -- feature lag_7 do ExtraTrees(BASE,j90) real (so usada AQUI, no
# benchmark -- os estratos novos nao tem lag_7, ver decisao na secao de features acima).
JANELA_EXTRATREES_BASE_D = 90   # mesma janela (j90) do ExtraTrees(BASE,j90) em experimento_valor.ipynb
JANELA_ABR2_QTD_D        = 45   # mesma janela (JANELA_DIAS) do ABR-2 em experimento_qtd.ipynb

query_fila_d = f"""
WITH cte_total AS (
    SELECT
        prop.propostaid,
        argMax(prop.propostaetapaid, prop.data) AS etapa,
        MAX(toDate(prop.data)) AS Data
    FROM crefazon15m.dbo_propostastatushistorico AS prop
    WHERE prop.data >= toDate(today() - INTERVAL {DIAS_QUERY_HIST} DAY) AND prop.data < today()
    GROUP BY prop.propostaid, toDate(prop.data)
)
SELECT tot.Data, COUNT(tot.propostaid) AS QUANTIDADE
FROM cte_total AS tot
WHERE tot.etapa = 15
GROUP BY tot.Data
"""

def _consultar_fila_d():
    _cli = clickhouse_connect.get_client(host=HOST, port=PORT, username=USER, password=PASS)
    return _cli.query_df(query_fila_d)

try:
    df_fila_d = carregar_com_cache('query_fila_diario_hierarquico',
                                   [pd.Timestamp.today().date(), DIAS_QUERY_HIST, query_fila_d], _consultar_fila_d)
    df_fila_d['Data'] = pd.to_datetime(df_fila_d['Data'], yearfirst=True)
    df_fila_d = df_fila_d.rename(columns={'Data': 'data', 'QUANTIDADE': 'fila'})
except RuntimeError as _e:
    print(f'[fila indisponivel] {_e} -- lag_7 do benchmark ExtraTrees(BASE,j90) ficara com fila=0')
    df_fila_d = pd.DataFrame(columns=['data', 'fila'])

serie_fila_total_d = (df_fila_d.set_index('data')['fila'].astype(float)
                      .reindex(serie_valor_total_d.index).fillna(0.0).sort_index())
print(f'series totais p/ benchmark de producao: {len(serie_valor_total_d)} dias uteis')

In [ ]:
# ==================== Receita real do ExtraTrees(BASE,j90) (valor) e do ABR-2 (qtd) --
# usadas TANTO no walk-forward 1 passo quanto na trajetoria (nowcast recursivo) =========
def _montar_feats_valor_d(serie_valor, serie_qntd, serie_fila, ate_data):
    """dia_semana + lag_0(qntd)..lag_6(valor) + lag_7(fila) -- identico a
    experimento_valor.ipynb / _montar_feats_valor (experimento_mensal.ipynb)."""
    _s = serie_valor.loc[:ate_data]
    _q = serie_qntd.reindex(_s.index)
    _f = serie_fila.reindex(_s.index) if serie_fila is not None else pd.Series(0.0, index=_s.index)
    _df = pd.DataFrame({'target': _s})
    _df['dia_semana'] = _df.index.dayofweek
    _df['lag_0'] = _q.shift(1)
    for _k in range(1, 7):
        _df[f'lag_{_k}'] = _s.shift(_k)
    _df['lag_7'] = _f.shift(1)
    return _df.dropna()


def _treinar_extratrees_base_d(serie_valor, serie_qntd, serie_fila, ate_data, janela=JANELA_EXTRATREES_BASE_D):
    """Refit do ExtraTrees(BASE,j90) real na janela rolante de `janela` dias -- mesmo
    hiperparametro (n_estimators=300, sem scaler) de experimento_valor.ipynb."""
    _tab = _montar_feats_valor_d(serie_valor, serie_qntd, serie_fila, ate_data).iloc[-janela:]
    if len(_tab) < 8:
        raise ValueError('historico insuficiente p/ ExtraTrees(BASE,j90)')
    _cols = ['dia_semana', 'lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7']
    _modelo = ExtraTreesRegressor(n_estimators=300, random_state=42)
    _modelo.fit(_tab[_cols], _tab['target'])
    return _modelo, _cols


def _recursivo_valor_d(modelo, cols, serie_valor, serie_qntd, serie_fila, data_origem, dias_alvo):
    """lag_1..lag_6 recursivos (cada previsao vira lag da proxima); lag_0(qntd)/lag_7(fila)
    CONGELADOS no ultimo valor real -- identico a _recursivo_valor (experimento_mensal.ipynb)."""
    _hist = list(serie_valor.loc[:data_origem].iloc[-6:].values.astype(float))
    _lag0 = float(serie_qntd.loc[:data_origem].iloc[-1])
    _lag7 = float(serie_fila.loc[:data_origem].iloc[-1]) if serie_fila is not None and len(serie_fila.loc[:data_origem]) else 0.0
    preds = []
    for _d in dias_alvo:
        _row = {'dia_semana': _d.weekday(), 'lag_0': _lag0}
        for _k in range(1, 7):
            _row[f'lag_{_k}'] = _hist[-_k]
        _row['lag_7'] = _lag7
        _X = pd.DataFrame([_row])[cols]
        _p = float(modelo.predict(_X)[0])
        preds.append(_p)
        _hist.append(_p)
    return preds


def _montar_feats_qtd_d(serie_qntd, serie_valor, ate_data):
    """dia_semana + lag_0(valor)..lag_6(qntd) -- identico a experimento_qtd.ipynb /
    _montar_feats_qtd (experimento_mensal.ipynb)."""
    _s = serie_qntd.loc[:ate_data]
    _v = serie_valor.reindex(_s.index)
    _df = pd.DataFrame({'target': _s})
    _df['dia_semana'] = _df.index.dayofweek
    _df['lag_0'] = _v.shift(1)
    for _k in range(1, 7):
        _df[f'lag_{_k}'] = _s.shift(_k)
    return _df.dropna()


def _treinar_abr2_d(pipeline_abr2, serie_qntd, serie_valor, ate_data, janela=JANELA_ABR2_QTD_D):
    """Refit do pipeline ABR-2 REAL (deepcopy) na janela rolante -- mesmo fallback de
    experimento_qtd.ipynb: tenta com todas as features, senao sem lag_0."""
    _tab = _montar_feats_qtd_d(serie_qntd, serie_valor, ate_data).iloc[-janela:]
    if len(_tab) < 8:
        raise ValueError('historico insuficiente p/ ABR-2')
    _cols_full = ['dia_semana', 'lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6']
    for _drop in ([], ['lag_0']):
        try:
            _cols = [c for c in _cols_full if c not in _drop]
            _p = copy.deepcopy(pipeline_abr2)
            _p.fit(_tab[_cols], _tab['target'])
            return _p, _cols
        except Exception:
            continue
    raise ValueError('ABR-2 nao conseguiu treinar (com ou sem lag_0)')


def _recursivo_qtd_d(modelo, cols, serie_qntd, serie_valor, data_origem, dias_alvo):
    """Mesma logica de _recursivo_valor_d p/ quantidade -- identico a _recursivo_qtd
    (experimento_mensal.ipynb)."""
    _hist = list(serie_qntd.loc[:data_origem].iloc[-6:].values.astype(float))
    _lag0 = float(serie_valor.loc[:data_origem].iloc[-1])
    preds = []
    for _d in dias_alvo:
        _row = {'dia_semana': _d.weekday(), 'lag_0': _lag0}
        for _k in range(1, 7):
            _row[f'lag_{_k}'] = _hist[-_k]
        _X = pd.DataFrame([_row])[cols]
        _p = float(modelo.predict(_X)[0])
        preds.append(_p)
        _hist.append(_p)
    return preds

In [ ]:
def wf_producao_valor_total_d(min_treino=MIN_TREINO_D, janela=JANELA_EXTRATREES_BASE_D):
    """Benchmark de producao (valor) p/ o walk-forward 1 PASSO: refit do
    ExtraTrees(BASE,j90) real em CADA fold (janela rolante), 1 dia a frente -- MESMA
    receita usada na trajetoria (_treinar_extratrees_base_d), so que sem recursao."""
    _tab = _montar_feats_valor_d(serie_valor_total_d, serie_qntd_total_d, serie_fila_total_d, serie_valor_total_d.index.max())
    _cols = ['dia_semana', 'lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7']
    rows = []
    for i in range(min_treino, len(_tab)):
        _tr = _tab.iloc[max(0, i - janela):i]
        _te = _tab.iloc[i]
        if len(_tr) < 8:
            continue
        _m = ExtraTreesRegressor(n_estimators=300, random_state=42)
        _m.fit(_tr[_cols], _tr['target'])
        _p = float(_m.predict(_te[_cols].values.reshape(1, -1))[0])
        rows.append({'data': _te.name, 'Previsto': _p, 'Realizado': float(_te['target'])})
    return pd.DataFrame(rows)


def wf_producao_qtd_total_d(min_treino=MIN_TREINO_D, janela=JANELA_ABR2_QTD_D):
    """Benchmark de producao (qtd) p/ o walk-forward 1 PASSO: refit do pipeline ABR-2
    real em CADA fold (janela rolante), 1 dia a frente."""
    if _PIPELINE_ABR2_QTD is None:
        return pd.DataFrame()
    _tab = _montar_feats_qtd_d(serie_qntd_total_d, serie_valor_total_d, serie_qntd_total_d.index.max())
    _cols_full = ['dia_semana', 'lag_0', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6']
    rows = []
    for i in range(min_treino, len(_tab)):
        _tr = _tab.iloc[max(0, i - janela):i]
        _te = _tab.iloc[i]
        if len(_tr) < 8:
            continue
        _p = None
        for _drop in ([], ['lag_0']):
            try:
                _cols = [c for c in _cols_full if c not in _drop]
                _mp = copy.deepcopy(_PIPELINE_ABR2_QTD)
                _mp.fit(_tr[_cols], _tr['target'])
                _p = float(_mp.predict(_te[_cols].values.reshape(1, -1))[0])
                break
            except Exception:
                continue
        if _p is None:
            continue
        rows.append({'data': _te.name, 'Previsto': _p, 'Realizado': float(_te['target'])})
    return pd.DataFrame(rows)


def nowcast_tabela_ml_recursivo_valor_d(ano, mes, min_treino_dias=60):
    """Benchmark de producao (valor) p/ a TRAJETORIA: identico a
    nowcast_tabela_ml_recursivo_valor (experimento_mensal.ipynb) -- refit do
    ExtraTrees(BASE,j90) real em CADA dia-origem, previsao RECURSIVA multi-passo."""
    _idx = pd.DatetimeIndex(serie_valor_total_d.index)
    _dias_mes = sorted(_idx[(_idx.year == ano) & (_idx.month == mes)])
    if not _dias_mes:
        return pd.DataFrame()
    _total_real = float(serie_valor_total_d.loc[_dias_mes].sum())
    rows = []
    for D in _dias_mes:
        _elapsed = [d for d in _dias_mes if d <= D]
        _remaining = [d for d in _dias_mes if d > D]
        _real_ate_D = float(serie_valor_total_d.loc[_elapsed].sum())
        _prev_rest = 0.0
        if _remaining:
            try:
                _modelo, _cols = _treinar_extratrees_base_d(serie_valor_total_d, serie_qntd_total_d, serie_fila_total_d, D)
                _preds = _recursivo_valor_d(_modelo, _cols, serie_valor_total_d, serie_qntd_total_d, serie_fila_total_d, D, _remaining)
                _prev_rest = float(sum(_preds))
            except Exception:
                _prev_rest = np.nan
        _nowcast_total = _real_ate_D + _prev_rest
        rows.append({'Dia Nowcast': pd.Timestamp(D).normalize(), 'Dias Corridos': len(_elapsed),
                     'Dias Restantes': len(_remaining), 'Nowcast': round(_nowcast_total, 1),
                     'Real': round(_total_real, 1),
                     'Erro %': (round((_nowcast_total - _total_real) / _total_real * 100, 2) if _total_real else np.nan)})
    return pd.DataFrame(rows)


def nowcast_tabela_ml_recursivo_qtd_d(ano, mes, min_treino_dias=60):
    """Benchmark de producao (qtd) p/ a TRAJETORIA: identico a
    nowcast_tabela_ml_recursivo_qtd (experimento_mensal.ipynb), com o ABR-2 real."""
    if _PIPELINE_ABR2_QTD is None:
        return pd.DataFrame()
    _idx = pd.DatetimeIndex(serie_qntd_total_d.index)
    _dias_mes = sorted(_idx[(_idx.year == ano) & (_idx.month == mes)])
    if not _dias_mes:
        return pd.DataFrame()
    _total_real = float(serie_qntd_total_d.loc[_dias_mes].sum())
    rows = []
    for D in _dias_mes:
        _elapsed = [d for d in _dias_mes if d <= D]
        _remaining = [d for d in _dias_mes if d > D]
        _real_ate_D = float(serie_qntd_total_d.loc[_elapsed].sum())
        _prev_rest = 0.0
        if _remaining:
            try:
                _modelo, _cols = _treinar_abr2_d(_PIPELINE_ABR2_QTD, serie_qntd_total_d, serie_valor_total_d, D)
                _preds = _recursivo_qtd_d(_modelo, _cols, serie_qntd_total_d, serie_valor_total_d, D, _remaining)
                _prev_rest = float(sum(_preds))
            except Exception:
                _prev_rest = np.nan
        _nowcast_total = _real_ate_D + _prev_rest
        rows.append({'Dia Nowcast': pd.Timestamp(D).normalize(), 'Dias Corridos': len(_elapsed),
                     'Dias Restantes': len(_remaining), 'Nowcast': round(_nowcast_total, 1),
                     'Real': round(_total_real, 1),
                     'Erro %': (round((_nowcast_total - _total_real) / _total_real * 100, 2) if _total_real else np.nan)})
    return pd.DataFrame(rows)

## Blend/Stacking e agregacao bottom-up da trajetoria (funcoes compartilhadas)

In [ ]:
def blend_nowcast_d(df_a, df_b, w=0.5):
    """Blend post-hoc de 2 trajetorias (Estimacao > Blend/Stacking): media ponderada de
    'Nowcast', alinhada por (Mes Previsto, Dia Nowcast)."""
    _m = df_a.merge(df_b, on=['Mes Previsto', 'Dia Nowcast'], suffixes=('_a', '_b'))
    _m['Nowcast'] = w * _m['Nowcast_a'] + (1 - w) * _m['Nowcast_b']
    _m['Real'] = _m['Real_a']
    _m['Dias Corridos'] = _m['Dias Corridos_a']
    _m['Dias Restantes'] = _m['Dias Restantes_a']
    _m['Razao'] = _m['Nowcast'] / _m['Real'].replace(0, np.nan)
    return _m[['Mes Previsto', 'Dia Nowcast', 'Dias Corridos', 'Dias Restantes', 'Nowcast', 'Real', 'Razao']]


def stacking_nowcast_d(df_a, df_b, min_meses=3):
    """Stacking post-hoc: Ridge(Nowcast_a, Nowcast_b) -> Real, treinado nos meses ja
    vistos (walk-forward por MES, nao por dia -- muito menos folds que o 1 passo)."""
    _m = df_a.merge(df_b, on=['Mes Previsto', 'Dia Nowcast'], suffixes=('_a', '_b')).sort_values(['Mes Previsto', 'Dia Nowcast'])
    _meses = list(dict.fromkeys(_m['Mes Previsto']))
    rows = []
    for _i, _mes in enumerate(_meses):
        if _i < min_meses:
            continue
        _tr = _m[_m['Mes Previsto'].isin(_meses[:_i])]
        _te = _m[_m['Mes Previsto'] == _mes]
        if _tr.empty or _te.empty:
            continue
        _mdl = Ridge(alpha=1.0)
        _mdl.fit(_tr[['Nowcast_a', 'Nowcast_b']].values, _tr['Real_a'].values)
        _p = _mdl.predict(_te[['Nowcast_a', 'Nowcast_b']].values)
        for (_, _r), _pi in zip(_te.iterrows(), _p):
            rows.append({'Mes Previsto': _r['Mes Previsto'], 'Dia Nowcast': _r['Dia Nowcast'],
                        'Dias Corridos': _r['Dias Corridos_a'], 'Dias Restantes': _r['Dias Restantes_a'],
                        'Nowcast': float(_pi), 'Real': _r['Real_a'],
                        'Razao': float(_pi) / _r['Real_a'] if _r['Real_a'] else np.nan})
    return pd.DataFrame(rows)


def nowcast_mlforecast_pooled_d(target, ano, mes, janela_meses, estratos=None, min_treino=MIN_TREINO_D):
    """Trajetoria -- Estrategia B: 1 modelo global pooled (ExtraTrees via mlforecast),
    refeito em CADA dia-origem D sobre TODAS as folhas cruzadas, prevendo os dias uteis
    restantes do mes de uma vez (horizonte NATIVO do mlforecast -- so a SOMA entra na
    trajetoria, nao importa o alinhamento exato de data por unique_id/step)."""
    _estratos = estratos if estratos is not None else ESTRATOS_VALIDOS_D
    _series = {chave: serie_diaria_estrato_d(chave, target) for chave in _estratos}
    _serie_total = sum(_series.values())
    _idx = pd.DatetimeIndex(_serie_total.index)
    _dias_mes = sorted(_idx[(_idx.year == ano) & (_idx.month == mes)])
    if not _dias_mes:
        return pd.DataFrame()
    _total_real = float(_serie_total.loc[_dias_mes].sum())
    _corte_hist = _idx.max() - pd.DateOffset(months=janela_meses)
    _modelos = {'ExtraTrees': ExtraTreesRegressor(n_estimators=300, max_depth=3, random_state=42)}
    rows = []
    for D in _dias_mes:
        _elapsed = [d for d in _dias_mes if d <= D]
        _remaining = [d for d in _dias_mes if d > D]
        _real_ate_D = float(_serie_total.loc[_elapsed].sum())
        _prev_rest = 0.0
        if _remaining:
            try:
                _paineis = []
                for _chave, _s in _series.items():
                    _s_ate_D = _s.loc[:D]
                    _s_ate_D = _s_ate_D[_s_ate_D.index >= _corte_hist]
                    if len(_s_ate_D) < min_treino + 5:
                        continue
                    _paineis.append(pd.DataFrame({'unique_id': _nome_estrato_d(_chave), 'ds': _s_ate_D.index, 'y': _s_ate_D.values}))
                _painel = pd.concat(_paineis, ignore_index=True)
                _mlf = MLForecast(models=_modelos, freq='D', lags=[1, 2, 3, 4, 5, 6],
                                  lag_transforms={1: [RollingMean(window_size=3)]}, date_features=['dayofweek'])
                _mlf.fit(_painel)
                _fc = _mlf.predict(h=len(_remaining))
                _prev_rest = float(_fc['ExtraTrees'].sum())
            except Exception:
                _prev_rest = np.nan
        _nowcast_total = _real_ate_D + _prev_rest
        rows.append({'Dia Nowcast': pd.Timestamp(D).normalize(), 'Dias Corridos': len(_elapsed),
                     'Dias Restantes': len(_remaining), 'Nowcast': round(_nowcast_total, 1),
                     'Real': round(_total_real, 1),
                     'Erro %': (round((_nowcast_total - _total_real) / _total_real * 100, 2) if _total_real else np.nan)})
    return pd.DataFrame(rows)


def _rank_trajetoria_estrato_d(nowcasts):
    """Ranking por trajetoria -- MESMO score composto do walk-forward 1 passo
    (calcular_score_d: RMSPE + % Outliers (MAPE>30%) + Vies % (sem modulo) + Desvio Padrao Erro %),
    calculado sobre a coluna 'Erro %' (Nowcast vs Real) de cada dia-origem/mes avaliado."""
    _linhas = {}
    for _nome, _df in nowcasts.items():
        if _df is None or _df.empty or 'Erro %' not in _df.columns:
            continue
        _e = pd.to_numeric(_df['Erro %'], errors='coerce').dropna()
        if _e.empty:
            continue
        _linhas[_nome] = {
            'RMSPE (%)':               round(float(np.sqrt(np.mean(np.square(_e)))), 2),
            '% Outliers (MAPE>30%)':   round(float((_e.abs() > 30).mean() * 100), 2),
            'Vies (%)':                round(float(_e.mean()), 2),
            'Desvio Padrao Erro (%)':  round(float(_e.std()), 2),
            'N':                       int(len(_e)),
        }
    if not _linhas:
        return pd.DataFrame()
    return calcular_score_d(pd.DataFrame(_linhas).T)


def agregar_estratos_trajetoria_d(nowcasts_por_estrato):
    """Estrategia A (trajetoria): cada estrato usa sua propria melhor trajetoria (menor
    Score composto -- ver _rank_trajetoria_estrato_d/calcular_score_d), soma Nowcast/Real
    por (Mes Previsto, Dia Nowcast) -- bottom-up."""
    _linhas, _vencedores = [], {}
    for _chave, _nowcasts in nowcasts_por_estrato.items():
        _rank = _rank_trajetoria_estrato_d(_nowcasts)
        if _rank.empty:
            continue
        _melhor = _rank.index[0]
        _vencedores[_nome_estrato_d(_chave)] = _melhor
        _d = _nowcasts[_melhor][['Mes Previsto', 'Dia Nowcast', 'Dias Corridos', 'Nowcast', 'Real']]
        _linhas.append(_d)
    if not _linhas:
        return pd.DataFrame(), _vencedores
    _todos = pd.concat(_linhas, ignore_index=True)
    _bu = _todos.groupby(['Mes Previsto', 'Dia Nowcast'], as_index=False).agg(
        **{'Dias Corridos': ('Dias Corridos', 'first'), 'Nowcast': ('Nowcast', 'sum'), 'Real': ('Real', 'sum')})
    _bu['Razao'] = _bu['Nowcast'] / _bu['Real'].replace(0, np.nan)
    return _bu.sort_values(['Mes Previsto', 'Dia Nowcast']).reset_index(drop=True), _vencedores

# Valor

## Previsao walkforward 1 passo (dia seguinte)

### Estimacao

In [ ]:
SERIES_VALOR_D = {chave: serie_diaria_estrato_d(chave, 'valor') for chave in ESTRATOS_VALIDOS_D}
TABS_VALOR_D   = {chave: tabela_supervisionada_estrato_d(s) for chave, s in SERIES_VALOR_D.items()}
DETS_VALOR_1P  = {chave: {} for chave in ESTRATOS_VALIDOS_D}   # acumula previsoes de TODAS as familias
print(f'{len(ESTRATOS_VALIDOS_D)} estratos | target=valor')

#### Baselines

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _tab = TABS_VALOR_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    DETS_VALOR_1P[_chave]['Naive'] = cache_modelo_d(
        'wf_naive_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _t=_tab: wf_baseline_estrato_d(_t, 'naive'))
    DETS_VALOR_1P[_chave]['MediaMovel5'] = cache_modelo_d(
        'wf_mm5_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _t=_tab: wf_baseline_estrato_d(_t, 'mm5'))
print('Baselines OK')

#### Modelos de regressao (sklearn)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _tab = TABS_VALOR_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    for _nome, _mk in _MODELOS_ARVORE_D.items():
        DETS_VALOR_1P[_chave][_nome] = cache_modelo_d(
            f'wf_regressao_1p_valor_{_nome}', [_nome_e, _versao_dados_d()],
            lambda _t=_tab, _m=_mk: wf_regressao_estrato_d(_t, _m))
print('Modelos de regressao OK')

#### Modelos estatisticos (statsforecast)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    _res_sf = cache_modelo_d(
        'wf_statsforecast_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _s=_serie: rodar_statsforecast_estrato_d(_s))
    DETS_VALOR_1P[_chave].update(_res_sf)
print('Modelos estatisticos OK')

#### RIPR (Ridge/Lasso/ElasticNet, formas funcionais)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _tab = TABS_VALOR_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    _res_ripr = cache_modelo_d(
        'wf_ripr_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _t=_tab: wf_ripr_estrato_d(_t))
    DETS_VALOR_1P[_chave].update(_res_ripr)
print('RIPR OK')

#### Kalman (local linear trend)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    DETS_VALOR_1P[_chave]['Kalman (local linear trend)'] = cache_modelo_d(
        'wf_kalman_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _s=_serie: wf_kalman_estrato_d(_s))
print('Kalman OK')

#### Blend/Stacking (estatistico + Naive)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _nome_e = _nome_estrato_d(_chave)
    _det_est = DETS_VALOR_1P[_chave].get('AutoARIMA (statsforecast)')
    _det_nai = DETS_VALOR_1P[_chave].get('Naive')
    if _det_est is None or _det_est.empty or _det_nai is None or _det_nai.empty:
        continue
    DETS_VALOR_1P[_chave]['Blend(ARIMA+Naive,w=0.5)'] = cache_modelo_d(
        'wf_blend_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _a=_det_est, _b=_det_nai: wf_blend_estrato_d(_a, _b, w=0.5))
    DETS_VALOR_1P[_chave]['Stacking(ARIMA+Naive)'] = cache_modelo_d(
        'wf_stacking_1p_valor', [_nome_e, _versao_dados_d()],
        lambda _a=_det_est, _b=_det_nai: wf_stacking_estrato_d(_a, _b))
print('Blend/Stacking OK')

#### mlforecast (Nixtla, modelo global pooled) -- Estrategia B

1 UNICO modelo global (cross-learning) sobre `ESTRATOS_VALIDOS_D`, testado com 6/12/24
meses de historico de treino (`JANELAS_HIST_MLFORECAST_MESES`) -- a janela vencedora e
usada como Estrategia B na Avaliacao abaixo.

In [ ]:
MELHOR_JANELA_VALOR_1P, RESULTADOS_MLFORECAST_VALOR_1P = cache_modelo_d(
    'mlforecast_pooled_1p_valor', [_versao_dados_d()],
    lambda: escolher_janela_mlforecast_d('valor'))
if MELHOR_JANELA_VALOR_1P:
    _jm, _nome_m = MELHOR_JANELA_VALOR_1P
    print(f'Melhor janela mlforecast pooled: {_jm} meses | modelo={_nome_m} | '
          f'RMSPE={RESULTADOS_MLFORECAST_VALOR_1P[MELHOR_JANELA_VALOR_1P]["rank"]}')
    for _k, _v in RESULTADOS_MLFORECAST_VALOR_1P.items():
        print(_k, '->', _v['rank'])
else:
    print('mlforecast pooled: sem resultado (historico insuficiente)')

### Avaliacao e Comparacao

In [ ]:
RANKINGS_VALOR_1P = {chave: montar_ranking_estrato_d(dets) for chave, dets in DETS_VALOR_1P.items()}

# Estrategia A -- melhor modelo POR estrato
BU_A_VALOR_1P, VENCEDORES_VALOR_1P = agregar_estratos_diario_d(DETS_VALOR_1P, RANKINGS_VALOR_1P)
RANK_A_VALOR_1P = _rank_linha_d(BU_A_VALOR_1P)
print('Estrategia A (melhor por estrato):', RANK_A_VALOR_1P)
print('Vencedores por estrato:', VENCEDORES_VALOR_1P)

In [ ]:
# Estrategia B -- melhor modelo UNICO p/ todas as series (mlforecast pooled, melhor janela)
if MELHOR_JANELA_VALOR_1P:
    BU_B_VALOR_1P = RESULTADOS_MLFORECAST_VALOR_1P[MELHOR_JANELA_VALOR_1P]['bu']
    RANK_B_VALOR_1P = RESULTADOS_MLFORECAST_VALOR_1P[MELHOR_JANELA_VALOR_1P]['rank']
else:
    BU_B_VALOR_1P, RANK_B_VALOR_1P = pd.DataFrame(), {}
print('Estrategia B (mlforecast pooled):', RANK_B_VALOR_1P)

In [ ]:
# Camada extra: reconciliacao hierarquica (BottomUp/MinTrace/ERM)
RECONC_VALOR_1P = cache_modelo_d(
    'reconciliacao_1p_valor', [_versao_dados_d()],
    lambda: rodar_reconciliacao_hierarquica_d('valor'))
for _nome, _det in RECONC_VALOR_1P.items():
    print(_nome, '->', _rank_linha_d(_det))

In [ ]:
# Benchmark de producao (Total) -- walk-forward 1 passo com a receita REAL de producao
DET_PRODUCAO_VALOR_1P = cache_modelo_d(
    'producao_1p_valor', [_versao_dados_d()],
    lambda: wf_producao_valor_total_d())
RANK_PRODUCAO_VALOR_1P = _rank_linha_d(DET_PRODUCAO_VALOR_1P)
print('Producao (ExtraTrees(BASE,j90)):', RANK_PRODUCAO_VALOR_1P)

In [ ]:
# Tabela final -- Estrategia A vs B vs reconciliacao vs producao real, ordenada por Score
_linhas_comp_valor = [{'Metodo': 'Producao (ExtraTrees(BASE,j90))', **RANK_PRODUCAO_VALOR_1P},
                    {'Metodo': 'Estrategia A (melhor por estrato)', **RANK_A_VALOR_1P}]
if RANK_B_VALOR_1P:
    _linhas_comp_valor.append({'Metodo': f'Estrategia B (mlforecast pooled, janela={MELHOR_JANELA_VALOR_1P})', **RANK_B_VALOR_1P})
for _nome, _det in RECONC_VALOR_1P.items():
    _linhas_comp_valor.append({'Metodo': _nome, **_rank_linha_d(_det)})

COMPARACAO_VALOR_1P = calcular_score_d(pd.DataFrame(_linhas_comp_valor).set_index('Metodo')).reset_index()
display(COMPARACAO_VALOR_1P)
_vencedor_valor = COMPARACAO_VALOR_1P.iloc[0]['Metodo']
_bate_valor = _vencedor_valor != 'Producao (ExtraTrees(BASE,j90))'
print(f'Bate a producao? {"SIM" if _bate_valor else "NAO"} -- vencedor (Score): {_vencedor_valor}')

### Export do melhor modelo

`MODELO_EXPORT_*_1P = None` usa o **vencedor automatico por Score** de cada estrato
(Estrategia A); troque pelo NOME de um modelo do roster (ex.: `'ExtraTrees'`,
`'AutoARIMA (statsforecast)'`, `'RIPR(Ridge,base)'`) p/ forcar esse modelo em todos os
estratos que o tiverem.

In [ ]:
MODELO_EXPORT_VALOR_1P = None   # None = melhor por Score (por estrato); ou nome de um modelo do roster

DETALHE_EXPORT_VALOR_1P = montar_export_1p_d(
    'valor', DETS_VALOR_1P, VENCEDORES_VALOR_1P, modelo_forcado=MODELO_EXPORT_VALOR_1P)
display(DETALHE_EXPORT_VALOR_1P.head(10))
_caminho_export_1p_valor = exportar_diario_hierarquico_d(
    DETALHE_EXPORT_VALOR_1P, 'diario_hierarquico_valor_1passo.xlsx')
print(f'Exportado: {_caminho_export_1p_valor} | {len(DETALHE_EXPORT_VALOR_1P)} linhas')

## Trajetoria do mes

Nowcast recursivo por estrato: a cada dia util D do mes, a previsao dos dias RESTANTES do
mes e refeita do zero (refit em D), somada ao realizado ate D. As mesmas 7 familias de
modelo entram, trocando so a fonte da previsao multi-passo (nativa p/
statsforecast/mlforecast, recursiva p/ sklearn/RIPR/baselines/Kalman -- ver builders na
secao `# ETL`).

### Estimacao

In [ ]:
MESES_COMPLETOS_VALOR = _meses_completos_serie_d(serie_valor_total_d)
NOWCAST_VALOR_TRAJ = {chave: {} for chave in ESTRATOS_VALIDOS_D}
print(f'Meses completos disponiveis p/ trajetoria (valor):', MESES_COMPLETOS_VALOR)

#### Baselines

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    NOWCAST_VALOR_TRAJ[_chave]['Naive'] = cache_modelo_d(
        'traj_naive_valor', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_baseline_d('naive'), a, m), _ms))
    NOWCAST_VALOR_TRAJ[_chave]['MediaMovel5'] = cache_modelo_d(
        'traj_mm5_valor', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_baseline_d('mm5'), a, m), _ms))
print('Baselines (trajetoria) OK')

#### Modelos de regressao (sklearn, recursivo)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    for _nome, _mk in _MODELOS_ARVORE_D.items():
        NOWCAST_VALOR_TRAJ[_chave][_nome] = cache_modelo_d(
            f'traj_regressao_valor_{_nome}', [_nome_e, _versao_dados_d()],
            lambda _s=_serie, _ms=_meses, _m=_mk: concatenar_nowcast_multi_mes_d(
                lambda a, mm, _s2=_s, _f=_builder_recursivo_sklearn_d(_m): nowcast_generico_estrato_d(_s2, _f, a, mm), _ms))
print('Modelos de regressao (trajetoria) OK')

#### Modelos estatisticos (statsforecast, horizonte nativo)

In [ ]:
_MODELOS_SF_TRAJ = [(AutoARIMA, {'season_length': 7}, 'AutoARIMA'), (AutoETS, {'season_length': 7}, 'AutoETS'),
                    (AutoCES, {'season_length': 7}, 'AutoCES'), (Theta, {'season_length': 7}, 'Theta')]
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    _modelos_sf = [CrostonSBA, TSB] if _eh_esparsa_d(_serie) else [m for m, _, _ in _MODELOS_SF_TRAJ]
    for _cls, _kw, _nome in _MODELOS_SF_TRAJ:
        if _cls not in _modelos_sf:
            continue
        NOWCAST_VALOR_TRAJ[_chave][f'{_nome} (statsforecast)'] = cache_modelo_d(
            f'traj_statsforecast_valor_{_nome}', [_nome_e, _versao_dados_d()],
            lambda _s=_serie, _ms=_meses, _c=_cls, _k=_kw: concatenar_nowcast_multi_mes_d(
                lambda a, mm, _s2=_s, _f=_builder_statsforecast_d(_c, **_k): nowcast_generico_estrato_d(_s2, _f, a, mm), _ms))
print('Modelos estatisticos (trajetoria) OK')

#### RIPR (Ridge com formas funcionais, recursivo)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    # Reusa a forma 'base' (mesmos FEATS_D) -- as formas log1p/interacao do 1-passo pedem
    # reconstruir a tabela supervisionada dentro do builder; Ridge(base) ja cobre a
    # comparacao RIPR-vs-resto na trajetoria sem multiplicar o custo de recursao por 3.
    NOWCAST_VALOR_TRAJ[_chave]['RIPR(Ridge,base)'] = cache_modelo_d(
        'traj_ripr_valor', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_recursivo_sklearn_d(_make_ridge), a, m), _ms))
print('RIPR (trajetoria) OK')

#### Kalman (local linear trend)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_VALOR_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    NOWCAST_VALOR_TRAJ[_chave]['Kalman (local linear trend)'] = cache_modelo_d(
        'traj_kalman_valor', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_kalman_d(), a, m), _ms))
print('Kalman (trajetoria) OK')

#### Blend/Stacking (estatistico + Naive)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _nome_e = _nome_estrato_d(_chave)
    _traj_est = NOWCAST_VALOR_TRAJ[_chave].get('AutoARIMA (statsforecast)')
    _traj_nai = NOWCAST_VALOR_TRAJ[_chave].get('Naive')
    if _traj_est is None or _traj_est.empty or _traj_nai is None or _traj_nai.empty:
        continue
    NOWCAST_VALOR_TRAJ[_chave]['Blend(ARIMA+Naive,w=0.5)'] = cache_modelo_d(
        'traj_blend_valor', [_nome_e, _versao_dados_d()],
        lambda _a=_traj_est, _b=_traj_nai: blend_nowcast_d(_a, _b, w=0.5))
    NOWCAST_VALOR_TRAJ[_chave]['Stacking(ARIMA+Naive)'] = cache_modelo_d(
        'traj_stacking_valor', [_nome_e, _versao_dados_d()],
        lambda _a=_traj_est, _b=_traj_nai: stacking_nowcast_d(_a, _b))
print('Blend/Stacking (trajetoria) OK')

#### mlforecast (Nixtla, modelo global pooled) -- Estrategia B

Reusa a melhor janela de historico ja encontrada no walk-forward 1 passo
(`MELHOR_JANELA_*_1P`) -- nao repete o teste de janela na trajetoria (custo alto:
recursivo por mes x dia-origem).

In [ ]:
_JM_VALOR_TRAJ = MELHOR_JANELA_VALOR_1P[0] if MELHOR_JANELA_VALOR_1P else 12
NOWCAST_POOLED_VALOR_TRAJ = cache_modelo_d(
    'traj_mlforecast_pooled_valor', [_JM_VALOR_TRAJ, _versao_dados_d()],
    lambda: concatenar_nowcast_multi_mes_d(
        lambda a, m: nowcast_mlforecast_pooled_d('valor', a, m, _JM_VALOR_TRAJ), MESES_COMPLETOS_VALOR))
print(f'mlforecast pooled (trajetoria, janela={_JM_VALOR_TRAJ} meses) OK | linhas={len(NOWCAST_POOLED_VALOR_TRAJ)}')

### Avaliacao e Comparacao

In [ ]:
# Estrategia A -- melhor trajetoria POR estrato (bottom-up)
BU_A_VALOR_TRAJ, VENCEDORES_VALOR_TRAJ = agregar_estratos_trajetoria_d(NOWCAST_VALOR_TRAJ)
print('Vencedores por estrato (trajetoria):', VENCEDORES_VALOR_TRAJ)
plot_nowcast_timeline_d(BU_A_VALOR_TRAJ, 'VALOR -- Estrategia A (melhor por estrato, bottom-up) -- timeline')
plot_nowcast_mediana_erro_d(BU_A_VALOR_TRAJ, 'VALOR -- Estrategia A (melhor por estrato, bottom-up)')

In [ ]:
# Estrategia B -- mlforecast pooled (bottom-up direto, ja e 1 modelo global)
plot_nowcast_timeline_d(NOWCAST_POOLED_VALOR_TRAJ, 'VALOR -- Estrategia B (mlforecast pooled) -- timeline')
plot_nowcast_mediana_erro_d(NOWCAST_POOLED_VALOR_TRAJ, 'VALOR -- Estrategia B (mlforecast pooled)')

In [ ]:
# Benchmark de producao (Total) -- nowcast recursivo com a receita REAL de producao
NOWCAST_PRODUCAO_VALOR_TRAJ = cache_modelo_d(
    'traj_producao_valor', [_versao_dados_d()],
    lambda: concatenar_nowcast_multi_mes_d(nowcast_tabela_ml_recursivo_valor_d, MESES_COMPLETOS_VALOR))
plot_nowcast_timeline_d(NOWCAST_PRODUCAO_VALOR_TRAJ, 'VALOR -- Producao (ExtraTrees(BASE,j90)) -- timeline')
plot_nowcast_mediana_erro_d(NOWCAST_PRODUCAO_VALOR_TRAJ, 'VALOR -- Producao (ExtraTrees(BASE,j90))')

#### Diagnostico por estrato (nowcast intramensal)

Mesmos 2 diagnosticos da secao "Nowcasting Intramensal" de `experimento_mensal.ipynb`,
aplicados aos estratos novos: `plot_nowcast_timeline_d` (timeline completa, varios meses
em sequencia, com a trajetoria do PROPRIO vencedor de cada estrato) e
`plot_nowcast_trajetoria_estrato_d` (zoom de UM mes, com banda +-5% do real).
`ESTRATOS_DIAGNOSTICO_D` limita a inspecao a alguns estratos (por padrao, os 3 primeiros
de `ESTRATOS_VALIDOS_D`) -- troque a lista pra inspecionar outros.

In [ ]:
ESTRATOS_DIAGNOSTICO_D = ESTRATOS_VALIDOS_D[:3]   # troque p/ inspecionar outros estratos

for _chave in ESTRATOS_DIAGNOSTICO_D:
    _nome_e = _nome_estrato_d(_chave)
    _venc = VENCEDORES_VALOR_TRAJ.get(_nome_e)
    if _venc is None:
        print(f'[diagnostico] {_nome_e}: sem vencedor de trajetoria (estrato sem meses completos avaliados)')
        continue
    _traj_e = NOWCAST_VALOR_TRAJ[_chave][_venc]
    plot_nowcast_timeline_d(_traj_e, f'{_nome_e} (VALOR) -- {_venc} -- timeline')
    _ultimo_mes = _traj_e['Mes Previsto'].iloc[-1] if not _traj_e.empty else None
    if _ultimo_mes:
        _df_1mes = _traj_e[_traj_e['Mes Previsto'] == _ultimo_mes]
        plot_nowcast_trajetoria_estrato_d(
            _df_1mes, f'{_nome_e} (VALOR) -- {_venc} -- nowcast intramensal {_ultimo_mes}')

In [ ]:
# Tabela final -- MESMO score composto (RMSPE + % Outliers + Vies % sem modulo + Desvio Padrao Erro %)
# calculado sobre a coluna 'Erro %' de cada trajetoria agregada
def _linha_score_traj(df):
    if df is None or df.empty or 'Nowcast' not in df.columns:
        return None
    _e = (pd.to_numeric(df['Nowcast'], errors='coerce') - pd.to_numeric(df['Real'], errors='coerce')) \
         / pd.to_numeric(df['Real'], errors='coerce').replace(0, np.nan) * 100
    _e = _e.dropna()
    if _e.empty:
        return None
    return {'RMSPE (%)': round(float(np.sqrt(np.mean(np.square(_e)))), 2),
            '% Outliers (MAPE>30%)': round(float((_e.abs() > 30).mean() * 100), 2),
            'Vies (%)': round(float(_e.mean()), 2),
            'Desvio Padrao Erro (%)': round(float(_e.std()), 2),
            'N': int(len(_e))}

_linhas_comp_traj_valor = {}
for _metodo, _df in [('Producao (ExtraTrees(BASE,j90))', NOWCAST_PRODUCAO_VALOR_TRAJ),
                     ('Estrategia A (melhor por estrato)', BU_A_VALOR_TRAJ),
                     ('Estrategia B (mlforecast pooled)', NOWCAST_POOLED_VALOR_TRAJ)]:
    _linha = _linha_score_traj(_df)
    if _linha is not None:
        _linhas_comp_traj_valor[_metodo] = _linha

COMPARACAO_VALOR_TRAJ = calcular_score_d(pd.DataFrame(_linhas_comp_traj_valor).T).reset_index(names='Metodo')
display(COMPARACAO_VALOR_TRAJ)
_vencedor_traj_valor = COMPARACAO_VALOR_TRAJ.iloc[0]['Metodo']
_bate_traj_valor = _vencedor_traj_valor != 'Producao (ExtraTrees(BASE,j90))'
print(f'Trajetoria bate a producao? {"SIM" if _bate_traj_valor else "NAO"} -- vencedor (Score): {_vencedor_traj_valor}')

### Export do melhor modelo

Mesma logica do export do walk-forward 1 passo, aplicada a trajetoria: `MODELO_EXPORT_*_TRAJ
= None` usa o vencedor automatico por Score de cada estrato; troque pelo NOME de um modelo
p/ forcar em todos os estratos que o tiverem.

In [ ]:
MODELO_EXPORT_VALOR_TRAJ = None   # None = melhor por Score (por estrato); ou nome de um modelo do roster

DETALHE_EXPORT_VALOR_TRAJ = montar_export_traj_d(
    'valor', NOWCAST_VALOR_TRAJ, VENCEDORES_VALOR_TRAJ, modelo_forcado=MODELO_EXPORT_VALOR_TRAJ)
display(DETALHE_EXPORT_VALOR_TRAJ.head(10))
_caminho_export_traj_valor = exportar_diario_hierarquico_d(
    DETALHE_EXPORT_VALOR_TRAJ, 'diario_hierarquico_valor_trajetoria.xlsx')
print(f'Exportado: {_caminho_export_traj_valor} | {len(DETALHE_EXPORT_VALOR_TRAJ)} linhas')

# Quantidade

## Previsao walkforward 1 passo (dia seguinte)

### Estimacao

In [ ]:
SERIES_QNTD_D = {chave: serie_diaria_estrato_d(chave, 'qntd') for chave in ESTRATOS_VALIDOS_D}
TABS_QNTD_D   = {chave: tabela_supervisionada_estrato_d(s) for chave, s in SERIES_QNTD_D.items()}
DETS_QNTD_1P  = {chave: {} for chave in ESTRATOS_VALIDOS_D}   # acumula previsoes de TODAS as familias
print(f'{len(ESTRATOS_VALIDOS_D)} estratos | target=qntd')

#### Baselines

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _tab = TABS_QNTD_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    DETS_QNTD_1P[_chave]['Naive'] = cache_modelo_d(
        'wf_naive_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _t=_tab: wf_baseline_estrato_d(_t, 'naive'))
    DETS_QNTD_1P[_chave]['MediaMovel5'] = cache_modelo_d(
        'wf_mm5_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _t=_tab: wf_baseline_estrato_d(_t, 'mm5'))
print('Baselines OK')

#### Modelos de regressao (sklearn)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _tab = TABS_QNTD_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    for _nome, _mk in _MODELOS_ARVORE_D.items():
        DETS_QNTD_1P[_chave][_nome] = cache_modelo_d(
            f'wf_regressao_1p_qntd_{_nome}', [_nome_e, _versao_dados_d()],
            lambda _t=_tab, _m=_mk: wf_regressao_estrato_d(_t, _m))
print('Modelos de regressao OK')

#### Modelos estatisticos (statsforecast)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    _res_sf = cache_modelo_d(
        'wf_statsforecast_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _s=_serie: rodar_statsforecast_estrato_d(_s))
    DETS_QNTD_1P[_chave].update(_res_sf)
print('Modelos estatisticos OK')

#### RIPR (Ridge/Lasso/ElasticNet, formas funcionais)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _tab = TABS_QNTD_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    _res_ripr = cache_modelo_d(
        'wf_ripr_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _t=_tab: wf_ripr_estrato_d(_t))
    DETS_QNTD_1P[_chave].update(_res_ripr)
print('RIPR OK')

#### Kalman (local linear trend)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _nome_e = _nome_estrato_d(_chave)
    DETS_QNTD_1P[_chave]['Kalman (local linear trend)'] = cache_modelo_d(
        'wf_kalman_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _s=_serie: wf_kalman_estrato_d(_s))
print('Kalman OK')

#### Blend/Stacking (estatistico + Naive)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _nome_e = _nome_estrato_d(_chave)
    _det_est = DETS_QNTD_1P[_chave].get('AutoARIMA (statsforecast)')
    _det_nai = DETS_QNTD_1P[_chave].get('Naive')
    if _det_est is None or _det_est.empty or _det_nai is None or _det_nai.empty:
        continue
    DETS_QNTD_1P[_chave]['Blend(ARIMA+Naive,w=0.5)'] = cache_modelo_d(
        'wf_blend_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _a=_det_est, _b=_det_nai: wf_blend_estrato_d(_a, _b, w=0.5))
    DETS_QNTD_1P[_chave]['Stacking(ARIMA+Naive)'] = cache_modelo_d(
        'wf_stacking_1p_qntd', [_nome_e, _versao_dados_d()],
        lambda _a=_det_est, _b=_det_nai: wf_stacking_estrato_d(_a, _b))
print('Blend/Stacking OK')

#### mlforecast (Nixtla, modelo global pooled) -- Estrategia B

1 UNICO modelo global (cross-learning) sobre `ESTRATOS_VALIDOS_D`, testado com 6/12/24
meses de historico de treino (`JANELAS_HIST_MLFORECAST_MESES`) -- a janela vencedora e
usada como Estrategia B na Avaliacao abaixo.

In [ ]:
MELHOR_JANELA_QNTD_1P, RESULTADOS_MLFORECAST_QNTD_1P = cache_modelo_d(
    'mlforecast_pooled_1p_qntd', [_versao_dados_d()],
    lambda: escolher_janela_mlforecast_d('qntd'))
if MELHOR_JANELA_QNTD_1P:
    _jm, _nome_m = MELHOR_JANELA_QNTD_1P
    print(f'Melhor janela mlforecast pooled: {_jm} meses | modelo={_nome_m} | '
          f'RMSPE={RESULTADOS_MLFORECAST_QNTD_1P[MELHOR_JANELA_QNTD_1P]["rank"]}')
    for _k, _v in RESULTADOS_MLFORECAST_QNTD_1P.items():
        print(_k, '->', _v['rank'])
else:
    print('mlforecast pooled: sem resultado (historico insuficiente)')

### Avaliacao e Comparacao

In [ ]:
RANKINGS_QNTD_1P = {chave: montar_ranking_estrato_d(dets) for chave, dets in DETS_QNTD_1P.items()}

# Estrategia A -- melhor modelo POR estrato
BU_A_QNTD_1P, VENCEDORES_QNTD_1P = agregar_estratos_diario_d(DETS_QNTD_1P, RANKINGS_QNTD_1P)
RANK_A_QNTD_1P = _rank_linha_d(BU_A_QNTD_1P)
print('Estrategia A (melhor por estrato):', RANK_A_QNTD_1P)
print('Vencedores por estrato:', VENCEDORES_QNTD_1P)

In [ ]:
# Estrategia B -- melhor modelo UNICO p/ todas as series (mlforecast pooled, melhor janela)
if MELHOR_JANELA_QNTD_1P:
    BU_B_QNTD_1P = RESULTADOS_MLFORECAST_QNTD_1P[MELHOR_JANELA_QNTD_1P]['bu']
    RANK_B_QNTD_1P = RESULTADOS_MLFORECAST_QNTD_1P[MELHOR_JANELA_QNTD_1P]['rank']
else:
    BU_B_QNTD_1P, RANK_B_QNTD_1P = pd.DataFrame(), {}
print('Estrategia B (mlforecast pooled):', RANK_B_QNTD_1P)

In [ ]:
# Camada extra: reconciliacao hierarquica (BottomUp/MinTrace/ERM)
RECONC_QNTD_1P = cache_modelo_d(
    'reconciliacao_1p_qntd', [_versao_dados_d()],
    lambda: rodar_reconciliacao_hierarquica_d('qntd'))
for _nome, _det in RECONC_QNTD_1P.items():
    print(_nome, '->', _rank_linha_d(_det))

In [ ]:
# Benchmark de producao (Total) -- walk-forward 1 passo com a receita REAL de producao
DET_PRODUCAO_QNTD_1P = cache_modelo_d(
    'producao_1p_qntd', [_versao_dados_d()],
    lambda: wf_producao_qtd_total_d())
RANK_PRODUCAO_QNTD_1P = _rank_linha_d(DET_PRODUCAO_QNTD_1P)
print('Producao (ABR-2):', RANK_PRODUCAO_QNTD_1P)

In [ ]:
# Tabela final -- Estrategia A vs B vs reconciliacao vs producao real, ordenada por Score
_linhas_comp_qntd = [{'Metodo': 'Producao (ABR-2)', **RANK_PRODUCAO_QNTD_1P},
                    {'Metodo': 'Estrategia A (melhor por estrato)', **RANK_A_QNTD_1P}]
if RANK_B_QNTD_1P:
    _linhas_comp_qntd.append({'Metodo': f'Estrategia B (mlforecast pooled, janela={MELHOR_JANELA_QNTD_1P})', **RANK_B_QNTD_1P})
for _nome, _det in RECONC_QNTD_1P.items():
    _linhas_comp_qntd.append({'Metodo': _nome, **_rank_linha_d(_det)})

COMPARACAO_QNTD_1P = calcular_score_d(pd.DataFrame(_linhas_comp_qntd).set_index('Metodo')).reset_index()
display(COMPARACAO_QNTD_1P)
_vencedor_qntd = COMPARACAO_QNTD_1P.iloc[0]['Metodo']
_bate_qntd = _vencedor_qntd != 'Producao (ABR-2)'
print(f'Bate a producao? {"SIM" if _bate_qntd else "NAO"} -- vencedor (Score): {_vencedor_qntd}')

### Export do melhor modelo

`MODELO_EXPORT_*_1P = None` usa o **vencedor automatico por Score** de cada estrato
(Estrategia A); troque pelo NOME de um modelo do roster (ex.: `'ExtraTrees'`,
`'AutoARIMA (statsforecast)'`, `'RIPR(Ridge,base)'`) p/ forcar esse modelo em todos os
estratos que o tiverem.

In [ ]:
MODELO_EXPORT_QNTD_1P = None   # None = melhor por Score (por estrato); ou nome de um modelo do roster

DETALHE_EXPORT_QNTD_1P = montar_export_1p_d(
    'qntd', DETS_QNTD_1P, VENCEDORES_QNTD_1P, modelo_forcado=MODELO_EXPORT_QNTD_1P)
display(DETALHE_EXPORT_QNTD_1P.head(10))
_caminho_export_1p_qntd = exportar_diario_hierarquico_d(
    DETALHE_EXPORT_QNTD_1P, 'diario_hierarquico_qntd_1passo.xlsx')
print(f'Exportado: {_caminho_export_1p_qntd} | {len(DETALHE_EXPORT_QNTD_1P)} linhas')

## Trajetoria do mes

Nowcast recursivo por estrato: a cada dia util D do mes, a previsao dos dias RESTANTES do
mes e refeita do zero (refit em D), somada ao realizado ate D. As mesmas 7 familias de
modelo entram, trocando so a fonte da previsao multi-passo (nativa p/
statsforecast/mlforecast, recursiva p/ sklearn/RIPR/baselines/Kalman -- ver builders na
secao `# ETL`).

### Estimacao

In [ ]:
MESES_COMPLETOS_QNTD = _meses_completos_serie_d(serie_qntd_total_d)
NOWCAST_QNTD_TRAJ = {chave: {} for chave in ESTRATOS_VALIDOS_D}
print(f'Meses completos disponiveis p/ trajetoria (qntd):', MESES_COMPLETOS_QNTD)

#### Baselines

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    NOWCAST_QNTD_TRAJ[_chave]['Naive'] = cache_modelo_d(
        'traj_naive_qntd', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_baseline_d('naive'), a, m), _ms))
    NOWCAST_QNTD_TRAJ[_chave]['MediaMovel5'] = cache_modelo_d(
        'traj_mm5_qntd', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_baseline_d('mm5'), a, m), _ms))
print('Baselines (trajetoria) OK')

#### Modelos de regressao (sklearn, recursivo)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    for _nome, _mk in _MODELOS_ARVORE_D.items():
        NOWCAST_QNTD_TRAJ[_chave][_nome] = cache_modelo_d(
            f'traj_regressao_qntd_{_nome}', [_nome_e, _versao_dados_d()],
            lambda _s=_serie, _ms=_meses, _m=_mk: concatenar_nowcast_multi_mes_d(
                lambda a, mm, _s2=_s, _f=_builder_recursivo_sklearn_d(_m): nowcast_generico_estrato_d(_s2, _f, a, mm), _ms))
print('Modelos de regressao (trajetoria) OK')

#### Modelos estatisticos (statsforecast, horizonte nativo)

In [ ]:
_MODELOS_SF_TRAJ = [(AutoARIMA, {'season_length': 7}, 'AutoARIMA'), (AutoETS, {'season_length': 7}, 'AutoETS'),
                    (AutoCES, {'season_length': 7}, 'AutoCES'), (Theta, {'season_length': 7}, 'Theta')]
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    _modelos_sf = [CrostonSBA, TSB] if _eh_esparsa_d(_serie) else [m for m, _, _ in _MODELOS_SF_TRAJ]
    for _cls, _kw, _nome in _MODELOS_SF_TRAJ:
        if _cls not in _modelos_sf:
            continue
        NOWCAST_QNTD_TRAJ[_chave][f'{_nome} (statsforecast)'] = cache_modelo_d(
            f'traj_statsforecast_qntd_{_nome}', [_nome_e, _versao_dados_d()],
            lambda _s=_serie, _ms=_meses, _c=_cls, _k=_kw: concatenar_nowcast_multi_mes_d(
                lambda a, mm, _s2=_s, _f=_builder_statsforecast_d(_c, **_k): nowcast_generico_estrato_d(_s2, _f, a, mm), _ms))
print('Modelos estatisticos (trajetoria) OK')

#### RIPR (Ridge com formas funcionais, recursivo)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    # Reusa a forma 'base' (mesmos FEATS_D) -- as formas log1p/interacao do 1-passo pedem
    # reconstruir a tabela supervisionada dentro do builder; Ridge(base) ja cobre a
    # comparacao RIPR-vs-resto na trajetoria sem multiplicar o custo de recursao por 3.
    NOWCAST_QNTD_TRAJ[_chave]['RIPR(Ridge,base)'] = cache_modelo_d(
        'traj_ripr_qntd', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_recursivo_sklearn_d(_make_ridge), a, m), _ms))
print('RIPR (trajetoria) OK')

#### Kalman (local linear trend)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _serie = SERIES_QNTD_D[_chave]
    _meses = _meses_completos_serie_d(_serie)
    _nome_e = _nome_estrato_d(_chave)
    NOWCAST_QNTD_TRAJ[_chave]['Kalman (local linear trend)'] = cache_modelo_d(
        'traj_kalman_qntd', [_nome_e, _versao_dados_d()],
        lambda _s=_serie, _ms=_meses: concatenar_nowcast_multi_mes_d(
            lambda a, m, _s2=_s: nowcast_generico_estrato_d(_s2, _builder_kalman_d(), a, m), _ms))
print('Kalman (trajetoria) OK')

#### Blend/Stacking (estatistico + Naive)

In [ ]:
for _chave in ESTRATOS_VALIDOS_D:
    _nome_e = _nome_estrato_d(_chave)
    _traj_est = NOWCAST_QNTD_TRAJ[_chave].get('AutoARIMA (statsforecast)')
    _traj_nai = NOWCAST_QNTD_TRAJ[_chave].get('Naive')
    if _traj_est is None or _traj_est.empty or _traj_nai is None or _traj_nai.empty:
        continue
    NOWCAST_QNTD_TRAJ[_chave]['Blend(ARIMA+Naive,w=0.5)'] = cache_modelo_d(
        'traj_blend_qntd', [_nome_e, _versao_dados_d()],
        lambda _a=_traj_est, _b=_traj_nai: blend_nowcast_d(_a, _b, w=0.5))
    NOWCAST_QNTD_TRAJ[_chave]['Stacking(ARIMA+Naive)'] = cache_modelo_d(
        'traj_stacking_qntd', [_nome_e, _versao_dados_d()],
        lambda _a=_traj_est, _b=_traj_nai: stacking_nowcast_d(_a, _b))
print('Blend/Stacking (trajetoria) OK')

#### mlforecast (Nixtla, modelo global pooled) -- Estrategia B

Reusa a melhor janela de historico ja encontrada no walk-forward 1 passo
(`MELHOR_JANELA_*_1P`) -- nao repete o teste de janela na trajetoria (custo alto:
recursivo por mes x dia-origem).

In [ ]:
_JM_QNTD_TRAJ = MELHOR_JANELA_QNTD_1P[0] if MELHOR_JANELA_QNTD_1P else 12
NOWCAST_POOLED_QNTD_TRAJ = cache_modelo_d(
    'traj_mlforecast_pooled_qntd', [_JM_QNTD_TRAJ, _versao_dados_d()],
    lambda: concatenar_nowcast_multi_mes_d(
        lambda a, m: nowcast_mlforecast_pooled_d('qntd', a, m, _JM_QNTD_TRAJ), MESES_COMPLETOS_QNTD))
print(f'mlforecast pooled (trajetoria, janela={_JM_QNTD_TRAJ} meses) OK | linhas={len(NOWCAST_POOLED_QNTD_TRAJ)}')

### Avaliacao e Comparacao

In [ ]:
# Estrategia A -- melhor trajetoria POR estrato (bottom-up)
BU_A_QNTD_TRAJ, VENCEDORES_QNTD_TRAJ = agregar_estratos_trajetoria_d(NOWCAST_QNTD_TRAJ)
print('Vencedores por estrato (trajetoria):', VENCEDORES_QNTD_TRAJ)
plot_nowcast_timeline_d(BU_A_QNTD_TRAJ, 'QUANTIDADE -- Estrategia A (melhor por estrato, bottom-up) -- timeline')
plot_nowcast_mediana_erro_d(BU_A_QNTD_TRAJ, 'QUANTIDADE -- Estrategia A (melhor por estrato, bottom-up)')

In [ ]:
# Estrategia B -- mlforecast pooled (bottom-up direto, ja e 1 modelo global)
plot_nowcast_timeline_d(NOWCAST_POOLED_QNTD_TRAJ, 'QUANTIDADE -- Estrategia B (mlforecast pooled) -- timeline')
plot_nowcast_mediana_erro_d(NOWCAST_POOLED_QNTD_TRAJ, 'QUANTIDADE -- Estrategia B (mlforecast pooled)')

In [ ]:
# Benchmark de producao (Total) -- nowcast recursivo com a receita REAL de producao
NOWCAST_PRODUCAO_QNTD_TRAJ = cache_modelo_d(
    'traj_producao_qntd', [_versao_dados_d()],
    lambda: concatenar_nowcast_multi_mes_d(nowcast_tabela_ml_recursivo_qtd_d, MESES_COMPLETOS_QNTD))
plot_nowcast_timeline_d(NOWCAST_PRODUCAO_QNTD_TRAJ, 'QUANTIDADE -- Producao (ABR-2) -- timeline')
plot_nowcast_mediana_erro_d(NOWCAST_PRODUCAO_QNTD_TRAJ, 'QUANTIDADE -- Producao (ABR-2)')

#### Diagnostico por estrato (nowcast intramensal)

Mesmos 2 diagnosticos da secao "Nowcasting Intramensal" de `experimento_mensal.ipynb`,
aplicados aos estratos novos: `plot_nowcast_timeline_d` (timeline completa, varios meses
em sequencia, com a trajetoria do PROPRIO vencedor de cada estrato) e
`plot_nowcast_trajetoria_estrato_d` (zoom de UM mes, com banda +-5% do real).
`ESTRATOS_DIAGNOSTICO_D` limita a inspecao a alguns estratos (por padrao, os 3 primeiros
de `ESTRATOS_VALIDOS_D`) -- troque a lista pra inspecionar outros.

In [ ]:
ESTRATOS_DIAGNOSTICO_D = ESTRATOS_VALIDOS_D[:3]   # troque p/ inspecionar outros estratos

for _chave in ESTRATOS_DIAGNOSTICO_D:
    _nome_e = _nome_estrato_d(_chave)
    _venc = VENCEDORES_QNTD_TRAJ.get(_nome_e)
    if _venc is None:
        print(f'[diagnostico] {_nome_e}: sem vencedor de trajetoria (estrato sem meses completos avaliados)')
        continue
    _traj_e = NOWCAST_QNTD_TRAJ[_chave][_venc]
    plot_nowcast_timeline_d(_traj_e, f'{_nome_e} (QNTD) -- {_venc} -- timeline')
    _ultimo_mes = _traj_e['Mes Previsto'].iloc[-1] if not _traj_e.empty else None
    if _ultimo_mes:
        _df_1mes = _traj_e[_traj_e['Mes Previsto'] == _ultimo_mes]
        plot_nowcast_trajetoria_estrato_d(
            _df_1mes, f'{_nome_e} (QNTD) -- {_venc} -- nowcast intramensal {_ultimo_mes}')

In [ ]:
# Tabela final -- MESMO score composto (RMSPE + % Outliers + Vies % sem modulo + Desvio Padrao Erro %)
# calculado sobre a coluna 'Erro %' de cada trajetoria agregada
def _linha_score_traj(df):
    if df is None or df.empty or 'Nowcast' not in df.columns:
        return None
    _e = (pd.to_numeric(df['Nowcast'], errors='coerce') - pd.to_numeric(df['Real'], errors='coerce')) \
         / pd.to_numeric(df['Real'], errors='coerce').replace(0, np.nan) * 100
    _e = _e.dropna()
    if _e.empty:
        return None
    return {'RMSPE (%)': round(float(np.sqrt(np.mean(np.square(_e)))), 2),
            '% Outliers (MAPE>30%)': round(float((_e.abs() > 30).mean() * 100), 2),
            'Vies (%)': round(float(_e.mean()), 2),
            'Desvio Padrao Erro (%)': round(float(_e.std()), 2),
            'N': int(len(_e))}

_linhas_comp_traj_qntd = {}
for _metodo, _df in [('Producao (ABR-2)', NOWCAST_PRODUCAO_QNTD_TRAJ),
                     ('Estrategia A (melhor por estrato)', BU_A_QNTD_TRAJ),
                     ('Estrategia B (mlforecast pooled)', NOWCAST_POOLED_QNTD_TRAJ)]:
    _linha = _linha_score_traj(_df)
    if _linha is not None:
        _linhas_comp_traj_qntd[_metodo] = _linha

COMPARACAO_QNTD_TRAJ = calcular_score_d(pd.DataFrame(_linhas_comp_traj_qntd).T).reset_index(names='Metodo')
display(COMPARACAO_QNTD_TRAJ)
_vencedor_traj_qntd = COMPARACAO_QNTD_TRAJ.iloc[0]['Metodo']
_bate_traj_qntd = _vencedor_traj_qntd != 'Producao (ABR-2)'
print(f'Trajetoria bate a producao? {"SIM" if _bate_traj_qntd else "NAO"} -- vencedor (Score): {_vencedor_traj_qntd}')

### Export do melhor modelo

Mesma logica do export do walk-forward 1 passo, aplicada a trajetoria: `MODELO_EXPORT_*_TRAJ
= None` usa o vencedor automatico por Score de cada estrato; troque pelo NOME de um modelo
p/ forcar em todos os estratos que o tiverem.

In [ ]:
MODELO_EXPORT_QNTD_TRAJ = None   # None = melhor por Score (por estrato); ou nome de um modelo do roster

DETALHE_EXPORT_QNTD_TRAJ = montar_export_traj_d(
    'qntd', NOWCAST_QNTD_TRAJ, VENCEDORES_QNTD_TRAJ, modelo_forcado=MODELO_EXPORT_QNTD_TRAJ)
display(DETALHE_EXPORT_QNTD_TRAJ.head(10))
_caminho_export_traj_qntd = exportar_diario_hierarquico_d(
    DETALHE_EXPORT_QNTD_TRAJ, 'diario_hierarquico_qntd_trajetoria.xlsx')
print(f'Exportado: {_caminho_export_traj_qntd} | {len(DETALHE_EXPORT_QNTD_TRAJ)} linhas')